# NIHONGA — Phase 1: Build the Prompt Dataset

**Phase registered:** `Phase 1 — Build the Prompt Dataset` (see [PLAN.md](PLAN.md#phase-1--build-the-prompt-dataset)).
**Experimental-order step:** `1. We build a 20k prompt dataset`.

## What this notebook does

1. Generates a **master pool of ~100k synthetic prompts** with structured templates and combinatorial generation.
   It follows the *capability structure* of Qwen-Image-Bench (alignment, visual quality, aesthetics, real-world
   fidelity, creative generation), but **no benchmark prompt is reused**.
2. Every prompt carries the metadata required by the plan
   (`prompt, dimension, subdimension, facet, difficulty, style, aspect_ratio, language`) plus traceability fields.
3. Uses **5 difficulty levels** and **5 prompting formats** (`short`, `tags`, `natural`, `descriptive`, `imperative`).
4. Assigns **stratified TRAIN / VALIDATION / INTERNAL TEST** splits (90/5/5) at pool level, so the pilot and the full
   100k share exactly the same split for every prompt. `QWEN-IMAGE-BENCH` is registered as a separate, evaluation-only split.
5. Selects the **20k pilot** as a stratified, nested prefix of the pool (the 100k expansion of Phase 15 is a superset).
6. **LLM rewriting** (run on Modal with vLLM): a fraction of the pilot is paraphrased in English and a fraction is
   translated into Chinese, with automatic constraint validation and fallback to the template prompt.
7. Serializes **everything** (config, full pool, pilot templates, raw LLM outputs, final splits, stats, manifest with hashes).

## Outputs (`data/phase1/`)

| File | Content |
|---|---|
| `config.json` | every parameter used for this run |
| `master_pool_100k.jsonl` | full template pool with `pool_rank` and `split` (**cached**: loaded, not recomputed, if present) |
| `../qwen_image_bench/prompts.jsonl` | the 1,000 bilingual Qwen-Image-Bench prompts (evaluation only) + `source.json` provenance |
| `decontamination_report.json` | overlap check of the pool against Qwen-Image-Bench (EN + ZH) |
| `pilot_20k/templates.jsonl` | pilot prompts before LLM rewriting (**cached**) |
| `pilot_20k/llm_raw.jsonl` | every raw LLM request/response + validation verdict (append-only, resumable) |
| `pilot_20k/llm_bench_rejections.json` | LLM outputs rejected for overlapping Qwen-Image-Bench |
| `pilot_20k/{train,validation,internal_test}.jsonl` | **final pilot dataset** |
| `splits_registry.json` | the 4 splits of the plan, incl. QWEN-IMAGE-BENCH (evaluation only) |
| `stats.json`, `manifest.json` | distribution statistics, file hashes, environment |

## 1. Configuration

Defines every tunable knob (seeds, sizes, weights, LLM settings, paths) and serializes it to `config.json`
so the run is fully reproducible.

In [20]:
# Phase 1 — configuration: every tunable parameter of the prompt-dataset build lives here
import json, hashlib, random, re, time, platform, sys, subprocess   # stdlib only: generation needs no GPU
from collections import Counter, defaultdict                        # counters for quotas and statistics
from datetime import datetime, timezone                             # timestamps for the manifest
from pathlib import Path                                            # filesystem paths

PHASE = "Phase 1 — Build the Prompt Dataset"         # phase registered in every serialized artifact
GENERATOR_VERSION = "phase1-v1"                      # bump whenever template logic changes (stored per record)
MASTER_SEED = 20260926                               # global seed -> the template pool is fully deterministic
MASTER_POOL_SIZE = 100_000                           # full target pool (Phase 15 expands to it later)
PILOT_SIZE = 20_000                                  # pilot built now (experimental order, step 1)
SPLIT_PATTERN = ["train"] * 18 + ["validation", "internal_test"]   # 90 / 5 / 5 applied inside every stratum

# Relative weight of each capability dimension in the pool
DIMENSION_WEIGHTS = {"alignment": 0.26, "visual_quality": 0.18, "aesthetics": 0.18,
                     "real_world": 0.18, "creative": 0.20}
# Sub-dimensions that get extra mass inside their dimension (default weight is 1.0)
SUBDIM_WEIGHT_OVERRIDES = {"counting": 1.3, "spatial_relation": 1.3, "text_rendering": 2.0}
# Difficulty distribution (1 = trivial single constraint, 5 = many interacting constraints)
DIFFICULTY_WEIGHTS = {1: 0.15, 2: 0.25, 3: 0.30, 4: 0.20, 5: 0.10}
# Sub-dimensions that are considered "hard / specialized" (used later by Phases 7 and 14 sampling)
HARD_SUBDIMS = {"counting", "spatial_relation", "multiple_objects", "composition_layout",
                "text_rendering", "fine_detail", "comics", "posters"}

# LLM rewriting (run on Modal) — applied to the pilot only
RUN_LLM_REWRITE = True                  # set False to build a template-only pilot without touching Modal
LLM_EN_REWRITE_FRACTION = 0.35          # share of (non-tag) pilot prompts paraphrased in English
LLM_ZH_TRANSLATE_FRACTION = 0.10        # share of pilot prompts translated to Simplified Chinese
LLM_MODEL = "Qwen/Qwen3-8B"             # open instruct model served with vLLM
LLM_GPU = "L40S"                        # Modal GPU type
LLM_BATCH_SIZE = 256                    # prompts per remote call
LLM_MAX_CONTAINERS = 2                  # parallel Modal containers
LLM_TEMPERATURE = 0.7                   # sampling temperature for paraphrase diversity

# Caching: expensive artifacts are never recomputed if already on disk
FORCE_RECOMPUTE = False                 # True -> regenerate the 100k pool and the 20k pilot, overwriting them

# Paths
DATA_DIR = Path("data/phase1")                                  # root of all Phase 1 artifacts
PILOT_DIR = DATA_DIR / f"pilot_{PILOT_SIZE // 1000}k"           # pilot artifacts
MASTER_POOL_PATH = DATA_DIR / f"master_pool_{MASTER_POOL_SIZE // 1000}k.jsonl"   # 100k pool (cached)
PILOT_TEMPLATES_PATH = PILOT_DIR / "templates.jsonl"            # 20k pilot before LLM rewriting (cached)
DATA_DIR.mkdir(parents=True, exist_ok=True)                     # create output folders
PILOT_DIR.mkdir(parents=True, exist_ok=True)

# Qwen-Image-Bench (EVALUATION ONLY: used here exclusively to decontaminate the synthetic pool)
BENCH_REPO = "Qwen/Qwen-Image-Bench"                            # Hugging Face dataset repo
BENCH_REVISION = "d2493deb153b020cf169c7e3f57d15e4dd697038"     # pinned commit -> reproducible benchmark
BENCH_FILE = "qwen_image_bench_hf_v0518.jsonl"                  # 1,000 prompts (prompt_cn / prompt_en) + model outputs
BENCH_DIR = Path("data/qwen_image_bench")                       # extracted benchmark artifacts
BENCH_PROMPTS_PATH = BENCH_DIR / "prompts.jsonl"                # extracted prompts (ID, prompt_en, prompt_cn, dims)
DECON_WORD_NGRAM = 6                                            # word n-gram size for English overlap
DECON_CHAR_NGRAM = 8                                            # character n-gram size for Chinese overlap
DECON_THRESHOLD = 0.5                                           # drop a prompt if >= 50% of its n-grams are in the bench
BENCH_DIR.mkdir(parents=True, exist_ok=True)

# Serialize the configuration (nothing can be lost). Explicit list: re-running this cell later must not pick up
# other UPPERCASE globals (e.g. GENERATORS / QUOTAS, which have tuple keys and are not JSON-serializable)
CONFIG_KEYS = ["PHASE", "GENERATOR_VERSION", "MASTER_SEED", "MASTER_POOL_SIZE", "PILOT_SIZE", "SPLIT_PATTERN",
               "DIMENSION_WEIGHTS", "SUBDIM_WEIGHT_OVERRIDES", "DIFFICULTY_WEIGHTS", "HARD_SUBDIMS",
               "RUN_LLM_REWRITE", "LLM_EN_REWRITE_FRACTION", "LLM_ZH_TRANSLATE_FRACTION", "LLM_MODEL", "LLM_GPU",
               "LLM_BATCH_SIZE", "LLM_MAX_CONTAINERS", "LLM_TEMPERATURE", "FORCE_RECOMPUTE", "DATA_DIR", "PILOT_DIR",
               "MASTER_POOL_PATH", "PILOT_TEMPLATES_PATH", "BENCH_REPO", "BENCH_REVISION", "BENCH_FILE", "BENCH_DIR",
               "BENCH_PROMPTS_PATH", "DECON_WORD_NGRAM", "DECON_CHAR_NGRAM", "DECON_THRESHOLD"]
CONFIG = {k: globals()[k] for k in CONFIG_KEYS}
CONFIG = {k: (sorted(v) if isinstance(v, set) else v.as_posix() if isinstance(v, Path) else v) for k, v in CONFIG.items()}
(DATA_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"{PHASE}\nConfig saved to {DATA_DIR / 'config.json'}")

Phase 1 — Build the Prompt Dataset
Config saved to data\phase1\config.json


## 2. Text utilities

Small grammar helpers (articles, plurals, number words, list joining), weighted sampling,
normalization for de-duplication and JSONL I/O helpers.

In [21]:
# Grammar + sampling + I/O helpers shared by all generators
NUM_WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight",
             "nine", "ten", "eleven", "twelve"]                                 # counts are written as words
IRREGULAR_PLURALS = {"sheep": "sheep", "goldfish": "goldfish", "fish": "fish", "deer": "deer",
                     "wolf": "wolves", "mouse": "mice", "goose": "geese", "potted cactus": "potted cacti",
                     "octopus": "octopuses", "person": "people", "tomato": "tomatoes", "leaf": "leaves",
                     "knife": "knives", "shelf": "shelves"}

def a(phrase):
    """Prefix a noun phrase with the right indefinite article."""
    w = phrase.lower()
    if w.startswith(("hour", "honest", "heir")):
        return "an " + phrase
    if w.startswith(("uni", "use", "eu", "one", "ukulele")):
        return "a " + phrase
    return ("an " if w[0] in "aeiou" else "a ") + phrase

def plural(noun):
    """Pluralize the head (last word) of a noun phrase."""
    for sing, plur in IRREGULAR_PLURALS.items():
        if noun == sing or noun.endswith(" " + sing):
            return noun[: len(noun) - len(sing)] + plur
    if re.search(r"[^aeiou]y$", noun):
        return noun[:-1] + "ies"
    if re.search(r"(s|x|z|ch|sh)$", noun):
        return noun + "es"
    return noun + "s"

def count_np(n, noun):
    """'a single apple' / 'three apples'."""
    return f"a single {noun}" if n == 1 else f"{NUM_WORDS[n]} {plural(noun)}"

def num_kw(n):
    """Keyword that must survive LLM rewriting for a count."""
    return "single" if n == 1 else NUM_WORDS[n]

def join_list(items):
    """'a', 'a and b', 'a, b and c'."""
    items = list(items)
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]

def cap(s):
    """Capitalize the first character only (keeps quoted text untouched)."""
    return s[:1].upper() + s[1:] if s else s

def weighted(rng, table):
    """Sample a key from a {key: weight} dict."""
    keys = list(table)
    return rng.choices(keys, weights=[table[k] for k in keys], k=1)[0]

def norm_key(text):
    """Normalization used for exact de-duplication."""
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s\"]", " ", text.lower())).strip()

def sha(text, n=12):
    """Short stable hash (ids, seeds)."""
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:n]

def write_jsonl(path, rows):
    """Write rows as UTF-8 JSONL and return the row count."""
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return len(rows)

def read_jsonl(path):
    """Read a UTF-8 JSONL file into a list of dicts."""
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

# Quick self-check of the grammar helpers
print(a("apple"), "|", a("unicorn"), "|", plural("red cherry"), "|", plural("wolf"), "|", count_np(3, "glass bottle"))

an apple | a unicorn | red cherries | wolves | three glass bottles


## 3. Shared vocabulary

Objects, animals, people, colors, shapes, materials, settings and generic "extras" (lighting, camera, mood, quality)
used by several generators. Larger vocabularies → more unique combinations → less template repetition.

In [22]:
# Shared vocabulary for the combinatorial generators
OBJECTS = ["apple", "orange", "lemon", "pear", "banana", "strawberry", "cherry", "teacup", "coffee mug", "book",
           "candle", "key", "pencil", "glass bottle", "vase", "alarm clock", "desk lamp", "backpack", "umbrella",
           "hat", "sneaker", "balloon", "rubber duck", "chess piece", "toy car", "teddy bear", "pumpkin",
           "sunflower", "rose", "cupcake", "donut", "egg", "seashell", "pine cone", "feather", "coin",
           "paper crane", "snow globe", "pocket watch", "tennis ball", "mushroom", "potted cactus", "lantern",
           "bell", "spoon", "bowl", "sock", "wine glass", "baseball cap", "toy robot", "paintbrush", "tea kettle",
           "lollipop", "tomato", "violin", "headphone set", "wooden block", "scarf", "rubber boot", "globe"]
ANIMALS = ["cat", "dog", "fox", "owl", "rabbit", "horse", "deer", "penguin", "parrot", "turtle", "frog", "elephant",
           "giraffe", "lion", "tiger", "panda", "koala", "squirrel", "hedgehog", "duck", "goldfish", "butterfly",
           "crow", "swan", "sheep", "wolf", "bear", "otter", "raccoon", "flamingo", "hummingbird", "crab", "snail",
           "ladybug", "goat", "zebra", "camel", "goose", "mouse", "octopus"]
LAND_ANIMALS = ["cat", "dog", "fox", "rabbit", "horse", "deer", "elephant", "giraffe", "lion", "tiger", "panda",
                "squirrel", "hedgehog", "sheep", "wolf", "bear", "raccoon", "goat", "zebra", "camel", "otter", "duck"]
PEOPLE = ["a young woman", "an elderly man", "a little girl", "a teenage boy", "a middle-aged woman", "a chef",
          "a firefighter", "a ballet dancer", "a street musician", "a scientist in a lab coat", "a farmer",
          "a skateboarder", "a grandmother", "a businessman in a suit", "a nurse in scrubs", "an astronaut",
          "a painter", "a fisherman", "a gardener", "a mechanic", "a young man", "an old woman", "a little boy",
          "a college student", "a postal worker", "a monk in saffron robes", "a cyclist in a yellow jersey"]
AGES = ["young", "middle-aged", "elderly", "teenage"]
ETHNICITIES = ["East Asian", "South Asian", "Southeast Asian", "Black", "Latino", "Middle Eastern", "white",
               "Indigenous", "Pacific Islander", "mixed-race"]
COLOR_BASIC = ["red", "blue", "green", "yellow", "orange", "purple", "pink", "white", "black", "gray", "brown"]
COLOR_RICH = ["teal", "turquoise", "crimson", "navy blue", "lime green", "magenta", "gold", "silver", "beige",
              "maroon", "lavender", "mint green", "coral", "mustard yellow", "burgundy", "cyan", "olive green",
              "ivory", "indigo"]
SHAPES = ["sphere", "cube", "cone", "cylinder", "pyramid", "torus", "hexagonal prism", "triangular prism",
          "octahedron", "capsule", "star-shaped block", "heart-shaped block", "disc", "crescent-shaped block",
          "dodecahedron", "ring"]
MATERIALS = ["wooden", "marble", "glass", "brushed steel", "copper", "golden", "ceramic", "velvet", "leather",
             "knitted wool", "concrete", "paper", "rubber", "jade", "bamboo", "crystal", "terracotta", "porcelain",
             "chrome", "cork", "stone", "denim", "silk", "carbon fiber", "obsidian", "brass", "felt", "ice"]
MATERIAL_OBJECTS = ["sphere", "teacup", "vase", "bowl", "chair", "cat statue", "spoon", "ring", "box",
                    "owl figurine", "lamp", "side table", "mask", "horse figurine", "clock", "bicycle", "shoe",
                    "apple", "rabbit sculpture", "guitar", "chess set", "helmet", "fish sculpture", "teapot"]
CONTAINERS = ["a glass jar", "a cardboard box", "a wicker basket", "a birdcage", "a large teacup", "a fishbowl",
              "a wooden crate", "an open suitcase", "a paper bag", "a ceramic bowl", "a metal bucket"]
TABLE_SETTINGS = ["on a wooden kitchen table", "on a white studio backdrop", "on a park bench", "on a sandy beach",
                  "on a desk next to a window", "on a grassy lawn", "on a marble countertop", "on snowy ground",
                  "on a rustic shelf", "in an empty room", "on a picnic blanket", "on a stone ledge",
                  "on a glass table", "on a red tablecloth", "on a concrete floor"]
ANIMAL_SETTINGS = ["in a grassy meadow", "on a snowy field", "on a sandy beach", "in a forest clearing",
                   "on a riverbank", "in a farmyard", "on a rocky hillside", "in a sunlit garden"]
SHAPE_SETTINGS = ["on a white table", "on a gray studio floor", "on a checkerboard floor", "on a pastel background",
                  "floating in empty space", "on a grassy lawn", "on a mirror surface", "on a sandy beach"]
SCENE_LOCATIONS = ["in a busy city street", "in a quiet forest", "in a sunlit meadow", "in a cozy café",
                   "in a snowy mountain village", "on a rainy sidewalk", "in a library", "in a desert canyon",
                   "at a harbor at dawn", "in a greenhouse", "in a subway station", "on a rooftop at dusk",
                   "in a small fishing village", "in a sunflower field", "in a night market"]
GENERAL_SUBJECTS = ["a lone tree", "a red lighthouse", "a person with a red umbrella", "a small wooden boat",
                    "a deer", "an empty park bench", "a cyclist", "a vintage car", "a hot air balloon", "a windmill",
                    "a cat sitting on a wall", "a dancer", "a fisherman", "a lone hiker", "a stone bridge",
                    "a thatched cottage", "a street lamp", "a white horse", "a small chapel", "a red bicycle",
                    "a paper lantern", "a heron", "a wooden pier", "a tram"]

# Coherent full scenes (subject + place) for aesthetic sub-dimensions
SCENES = ["a lighthouse on a rocky cliff", "a fisherman on a wooden pier", "a cyclist on a country road",
          "a small wooden boat on a calm lake", "a deer in a misty forest", "a thatched cottage in a meadow",
          "a street musician in a busy square", "a vintage car on a coastal road", "a windmill in a tulip field",
          "a cat sitting on a garden wall", "a hot air balloon over rolling hills", "a dancer in an empty theater",
          "a tram on a hillside street", "a heron standing in a rice paddy", "a stone bridge over a river",
          "a lone hiker on a mountain ridge", "a café terrace on a cobblestone street", "a red bicycle against a brick wall",
          "a small chapel on a hill", "a market stall full of fruit", "a white horse in a snowy field",
          "a woman reading on a park bench", "a paper lantern hanging over a canal", "an old man feeding pigeons"]

# Generic optional extras — appended depending on difficulty and style
LIGHTING_OPT = ["soft morning light", "golden hour sunlight", "overcast diffuse light", "warm window light",
                "dramatic side lighting", "soft studio lighting", "late afternoon sun", "gentle backlight"]
CAMERA_OPT = ["shallow depth of field", "35mm lens", "85mm lens", "wide-angle lens", "bokeh background",
              "50mm lens", "deep depth of field"]
MOOD_OPT = ["calm atmosphere", "cheerful mood", "nostalgic feeling", "mysterious atmosphere", "serene mood",
            "melancholic tone", "whimsical mood", "tense atmosphere"]
QUALITY_OPT = ["highly detailed", "sharp focus", "clean composition", "rich colors", "soft shadows",
               "high dynamic range", "crisp details", "natural colors"]
OPTIONAL_POOLS = {"lighting": LIGHTING_OPT, "camera": CAMERA_OPT, "mood": MOOD_OPT, "quality": QUALITY_OPT}
DIM_OPTIONAL_POOLS = {"alignment": ["quality"], "visual_quality": ["camera", "quality"],  # one extra per category max
                      "aesthetics": ["camera", "quality"], "real_world": ["lighting", "camera", "quality", "mood"],
                      "creative": ["mood", "quality", "lighting"]}

def color(rng, d):
    """Basic colors for easy prompts, increasingly nuanced colors as difficulty grows."""
    return rng.choice(COLOR_RICH if rng.random() < 0.15 * d else COLOR_BASIC)

def person(rng):
    """A person description with explicit age/ethnicity diversity."""
    age = rng.choice(AGES)
    who = rng.choice(["girl", "boy"]) if age == "teenage" else rng.choice(["woman", "man", "person"])
    return a(f"{age} {rng.choice(ETHNICITIES)} {who}")

print(f"{len(OBJECTS)} objects, {len(ANIMALS)} animals, {len(PEOPLE)} people, "
      f"{len(COLOR_BASIC) + len(COLOR_RICH)} colors, {len(SHAPES)} shapes, {len(MATERIALS)} materials")

60 objects, 40 animals, 27 people, 30 colors, 16 shapes, 28 materials


## 4. Visual styles, prompt formats and aspect ratios

* **`style`** (metadata) = the visual medium (photographic, watercolor, nihonga, ...). Each style has a few surface
  variants to avoid repeating the exact same phrase.
* **`prompt_format`** = the *prompting style* (how the user writes): `short`, `tags`, `natural`, `descriptive`,
  `imperative`. Harder prompts favour longer formats.
* **`aspect_ratio`** is sampled per sub-dimension (portraits → vertical, landscapes → wide, ...).
* `render()` turns a generator output (core description + mandatory extras) into the final prompt text.

In [23]:
# Visual styles: key -> list of (noun phrase used in sentences, tag string used in tag-style prompts)
STYLES = {
    "photographic": [("a photorealistic photograph", "photorealistic, professional photography"),
                     ("a 35mm film photograph", "35mm film photo, film grain"),
                     ("a high-resolution DSLR photo", "DSLR photo, high resolution"),
                     ("a candid photograph", "candid photo, natural light")],
    "cinematic": [("a cinematic film still", "cinematic still, anamorphic, color graded"),
                  ("a movie still", "movie still, dramatic lighting")],
    "digital_illustration": [("a digital illustration", "digital illustration, clean lines"),
                             ("a digital painting", "digital painting, painterly")],
    "watercolor": [("a watercolor painting", "watercolor, soft washes, paper texture")],
    "oil_painting": [("an oil painting", "oil on canvas, visible brushstrokes"),
                     ("a classical oil painting", "classical oil painting, old master style")],
    "anime": [("an anime-style illustration", "anime style, cel shading")],
    "3d_render": [("a 3D render", "3D render, soft global illumination"),
                  ("a stylized 3D render", "stylized 3D, clay-like materials")],
    "pixel_art": [("a pixel art image", "pixel art, 16-bit")],
    "ink_wash": [("a traditional ink wash painting", "sumi-e ink wash, rice paper")],
    "ukiyo_e": [("a ukiyo-e woodblock print", "ukiyo-e woodblock print, flat colors")],
    "nihonga": [("a nihonga painting", "nihonga, mineral pigments on silk, gold leaf accents")],
    "pencil_sketch": [("a pencil sketch", "graphite pencil sketch, cross-hatching")],
    "flat_vector": [("a flat vector illustration", "flat vector art, bold simple shapes")],
    "claymation": [("a claymation scene", "claymation, handmade clay figures")],
    "low_poly": [("a low-poly 3D illustration", "low poly, faceted geometry")],
    "comic_book": [("a comic book illustration", "comic book style, bold ink outlines, halftone")],
    "art_nouveau": [("an Art Nouveau illustration", "art nouveau, ornamental linework")],
    "isometric": [("an isometric illustration", "isometric view, clean 3D illustration")],
    "childrens_book": [("a children's book illustration", "children's book illustration, gouache")],
    "concept_art": [("a piece of concept art", "concept art, matte painting, highly detailed")],
    "graphic_design": [("a graphic design layout", "graphic design, clean typography, print layout")],
    "vintage_print": [("a vintage screen-printed poster", "vintage print, limited palette, risograph texture")],
}
PHOTO_STYLES = {"photographic", "cinematic"}   # camera extras only make sense for these

# Default style distribution per dimension
DIM_STYLE_POOL = {
    "alignment": {"photographic": .35, "3d_render": .20, "digital_illustration": .15, "flat_vector": .07,
                  "watercolor": .07, "oil_painting": .05, "pencil_sketch": .04, "anime": .04, "claymation": .03},
    "visual_quality": {"photographic": .70, "3d_render": .10, "cinematic": .10, "oil_painting": .05,
                       "watercolor": .05},
    "aesthetics": {"photographic": .40, "cinematic": .20, "oil_painting": .10, "watercolor": .10,
                   "digital_illustration": .10, "nihonga": .05, "ink_wash": .05},
    "real_world": {"photographic": .75, "cinematic": .10, "3d_render": .05, "watercolor": .05, "isometric": .05},
    "creative": {"digital_illustration": .25, "concept_art": .15, "comic_book": .10, "anime": .10,
                 "3d_render": .10, "flat_vector": .10, "photographic": .10, "oil_painting": .05,
                 "childrens_book": .05},
}
# Sub-dimension specific style distributions
SUBDIM_STYLE_POOL = {
    "text_rendering": {"photographic": .50, "graphic_design": .25, "3d_render": .10, "flat_vector": .10,
                       "vintage_print": .05},
    "posters": {"graphic_design": .35, "vintage_print": .25, "flat_vector": .15, "digital_illustration": .15,
                "art_nouveau": .10},
    "comics": {"comic_book": .60, "anime": .20, "flat_vector": .10, "childrens_book": .10},
    "cinematic_framing": {"cinematic": .60, "photographic": .40},
    "skin": {"photographic": .85, "cinematic": .10, "oil_painting": .05},
    "concept_art": {"concept_art": .60, "digital_illustration": .25, "3d_render": .15},
    "character_design": {"digital_illustration": .40, "anime": .25, "concept_art": .15, "3d_render": .10,
                         "flat_vector": .10},
}

# Prompting formats by difficulty (longer, more structured formats for harder prompts)
FORMAT_BY_DIFFICULTY = {
    1: {"short": .45, "natural": .30, "tags": .15, "imperative": .10},
    2: {"short": .25, "natural": .40, "tags": .15, "imperative": .10, "descriptive": .10},
    3: {"short": .10, "natural": .40, "tags": .15, "imperative": .10, "descriptive": .25},
    4: {"natural": .30, "tags": .10, "imperative": .15, "descriptive": .45},
    5: {"natural": .20, "imperative": .20, "descriptive": .60},
}
EXTRAS_BY_DIFFICULTY = {1: (0, 0), 2: (0, 1), 3: (1, 1), 4: (1, 2), 5: (2, 3)}   # optional extras (min, max)
IMPERATIVE_VERBS = ["Generate an image of", "Create an image of", "Make a picture of", "Show", "Render"]

# Aspect ratios
DEFAULT_AR = {"1:1": .40, "16:9": .15, "9:16": .10, "4:3": .12, "3:4": .10, "3:2": .07, "2:3": .06}
VERTICAL_AR = {"3:4": .35, "2:3": .30, "9:16": .20, "1:1": .15}
WIDE_AR = {"16:9": .45, "3:2": .25, "4:3": .15, "1:1": .10, "21:9": .05}
SUBDIM_AR = {"portraits": VERTICAL_AR, "skin": VERTICAL_AR, "character_design": VERTICAL_AR, "posters": VERTICAL_AR,
             "clothing": VERTICAL_AR, "landscapes": WIDE_AR, "cinematic_framing": WIDE_AR, "architecture": WIDE_AR,
             "concept_art": WIDE_AR, "interiors": WIDE_AR}

def render(rng, core, extras, style_key, fmt):
    """Turn a core description + extras into a prompt in the requested prompting format."""
    noun, tags = rng.choice(STYLES[style_key])
    core = core.strip().rstrip(".")
    ext = ", ".join(extras)
    if fmt == "short":
        return cap(core) + (f", {ext}" if ext else "") + f", {tags.split(',')[0]}"
    if fmt == "tags":
        return ", ".join([core] + list(extras) + [tags])
    if fmt == "natural":
        return f"{cap(noun)} of {core}" + (f", {ext}" if ext else "") + "."
    if fmt == "descriptive":
        sentences = [f"{cap(noun)} of {core}."] + [cap(e) + "." for e in extras] + [cap(tags) + "."]
        return " ".join(sentences)
    if fmt == "imperative":
        return f"{rng.choice(IMPERATIVE_VERBS)} {core}" + (f", {ext}" if ext else "") + f". Style: {tags}."
    raise ValueError(fmt)

print(f"{len(STYLES)} visual styles, {len(FORMAT_BY_DIFFICULTY[3])} formats at difficulty 3")
print(render(random.Random(0), "three red apples on a wooden table", ["soft morning light"], "watercolor", "natural"))

22 visual styles, 5 formats at difficulty 3
A watercolor painting of three red apples on a wooden table, soft morning light.


## 5. Generators — Prompt alignment

Each generator is `g(rng, difficulty) -> dict(core, facet, keywords, [extras], [style], [ar])`:

* `core` — the main description,
* `facet` — the fine-grained capability tested (e.g. `inside_left_of`, `count_7`, `binding_3`),
* `keywords` — tokens that **must survive** LLM rewriting (numbers, colors, relations),
* `extras` — mandatory modifiers (they are never dropped),
* optional `style` / `ar` overrides.

Difficulty increases the number of constraints (objects, counts, attribute bindings, chained relations).

In [24]:
# ---------------- alignment / counting ----------------
def g_counting(rng, d):
    """Exact counts; d>=4 mixes several object types with separate counts."""
    n = {1: (1, 3), 2: (2, 5), 3: (4, 7), 4: (6, 9), 5: (8, 12)}[d]
    n = rng.randint(*n)
    animals = rng.random() < 0.3
    pool, setting = (ANIMALS, rng.choice(ANIMAL_SETTINGS)) if animals else (OBJECTS, rng.choice(TABLE_SETTINGS))
    if d >= 4 and rng.random() < 0.5:                                         # multi-type counting
        k = 2 if d == 4 else rng.choice([2, 3])
        nouns, counts = rng.sample(pool, k), [rng.randint(1, 5) for _ in range(k)]
        parts = [count_np(c, x) for c, x in zip(counts, nouns)]
        return dict(core=f"exactly {join_list(parts)} {setting}", facet=f"multi_count_{k}types",
                    keywords=[num_kw(c) for c in counts])
    noun = rng.choice(pool)
    if d >= 2 and not animals and rng.random() < 0.6:
        noun = f"{color(rng, d)} {noun}"
    return dict(core=f"{count_np(n, noun)} {setting}", facet=f"count_{n}", keywords=[num_kw(n)])

# ---------------- alignment / colors ----------------
COUNTERFACTUAL = [("banana", "yellow"), ("strawberry", "red"), ("lemon", "yellow"), ("carrot", "orange"),
                  ("flamingo", "pink"), ("polar bear", "white"), ("pumpkin", "orange"), ("swan", "white"),
                  ("crow", "black"), ("tomato", "red"), ("frog", "green"), ("school bus", "yellow"),
                  ("fire truck", "red"), ("broccoli", "green"), ("panda", "black and white")]

def g_colors(rng, d):
    """Color attribute binding: single color -> 4 bound pairs, patterns and counterfactual colors."""
    objs = rng.sample(OBJECTS, 4)
    cs = rng.sample(COLOR_BASIC if d < 3 else COLOR_BASIC + COLOR_RICH, 4)
    setting = rng.choice(TABLE_SETTINGS)
    bound = [a(f"{c} {o}") for c, o in zip(cs, objs)]
    if d == 1:
        return dict(core=bound[0], facet="single_color", keywords=cs[:1])
    if d == 2:
        return dict(core=f"{bound[0]} on a plain {cs[1]} background", facet="object_background", keywords=cs[:2])
    if d == 3:
        return dict(core=f"{bound[0]} and {bound[1]} {setting}", facet="binding_2", keywords=cs[:2])
    if d == 4:
        if rng.random() < 0.5:
            return dict(core=f"{join_list(bound[:3])} {setting}", facet="binding_3", keywords=cs[:3])
        return dict(core=f"{a(objs[0])} with {cs[0]} and {cs[1]} stripes next to {bound[2]}",
                    facet="pattern_binding", keywords=cs[:3])
    if rng.random() < 0.5:
        return dict(core=f"{join_list(bound)} {setting}", facet="binding_4", keywords=cs)
    noun, natural = rng.choice(COUNTERFACTUAL)
    wrong = rng.choice([c for c in COLOR_BASIC + COLOR_RICH if c not in natural])
    return dict(core=f"{a(wrong + ' ' + noun)} next to {bound[1]} {setting}", facet="counterfactual_color",
                keywords=[wrong, cs[1]])

# ---------------- alignment / shapes ----------------
def shape_kw(s):
    """Head keyword of a shape name ('star-shaped block' -> 'star')."""
    return re.split(r"[- ]", s)[0]

def g_shapes(rng, d):
    """Geometric shapes with colors/materials, rows and stacks."""
    s, c = rng.sample(SHAPES, 3), rng.sample(COLOR_BASIC, 3)
    m, setting = rng.choice(MATERIALS), rng.choice(SHAPE_SETTINGS)
    items = [a(f"{ci} {si}") for ci, si in zip(c, s)]
    if d == 1:
        return dict(core=f"{a(s[0])} {setting}", facet="single_shape", keywords=[shape_kw(s[0])])
    if d == 2:
        return dict(core=f"{items[0]} {setting}", facet="shape_color", keywords=[c[0], shape_kw(s[0])])
    if d == 3:
        return dict(core=f"{a(m + ' ' + s[0])} next to {items[1]} {setting}", facet="shape_pair",
                    keywords=[m.split()[-1], shape_kw(s[0]), c[1], shape_kw(s[1])])
    kws = c + [shape_kw(x) for x in s]
    if d == 4:
        return dict(core=f"{join_list(items)} arranged in a row from left to right {setting}", facet="shape_row",
                    keywords=kws)
    return dict(core=f"a stack of three shapes {setting}: {items[0]} at the bottom, {items[1]} in the middle "
                     f"and {items[2]} on top", facet="shape_stack", keywords=kws)

# ---------------- alignment / materials ----------------
def g_materials(rng, d):
    """Material rendering of objects: single, pairs, split materials, triplets."""
    m, o = rng.sample(MATERIALS, 3), rng.sample(MATERIAL_OBJECTS, 3)
    setting = rng.choice(TABLE_SETTINGS)
    kw = [x.split()[-1] for x in m]
    if d == 1:
        return dict(core=a(f"{m[0]} {o[0]}"), facet="single_material", keywords=kw[:1])
    if d == 2:
        return dict(core=f"{a(m[0] + ' ' + o[0])} {setting}", facet="single_material_scene", keywords=kw[:1])
    if d == 3:
        return dict(core=f"{a(m[0] + ' ' + o[0])} beside {a(m[1] + ' ' + o[1])} {setting}", facet="material_pair",
                    keywords=kw[:2])
    if d == 4:
        return dict(core=f"{a(o[0])} that is half {m[0]} and half {m[1]} {setting}", facet="material_split",
                    keywords=kw[:2])
    return dict(core=f"{join_list([a(mi + ' ' + oi) for mi, oi in zip(m, o)])} lined up {setting}",
                facet="material_triplet", keywords=kw)

# ---------------- alignment / multiple objects ----------------
def g_multiple_objects(rng, d):
    """d+1 distinct objects (2..6), with optional colors from d>=3."""
    n = d + 1
    pool = OBJECTS + ANIMALS if rng.random() < 0.4 else OBJECTS
    nouns = rng.sample(pool, n)
    if d >= 3:
        nouns = [f"{color(rng, d)} {x}" if rng.random() < 0.5 else x for x in nouns]
    setting = rng.choice(TABLE_SETTINGS + SCENE_LOCATIONS)
    return dict(core=f"{join_list([a(x) for x in nouns])} {setting}", facet=f"objects_{n}",
                keywords=[x.split()[-1] for x in nouns])

# ---------------- alignment / actions ----------------
HUMAN_ACTIONS = [("reading a book", None), ("riding a bicycle", "along a canal"), ("playing the violin", None),
                 ("cooking pasta", "in a home kitchen"), ("jumping over a puddle", "on a rainy street"),
                 ("painting on an easel", None), ("throwing a frisbee", "in a park"),
                 ("climbing a rock wall", "at an indoor climbing gym"), ("watering plants", "in a greenhouse"),
                 ("typing on a laptop", None), ("juggling three oranges", None), ("pouring tea", None),
                 ("flying a kite", "on a windy hilltop"), ("taking a photo with a vintage camera", None),
                 ("carrying a stack of boxes", None), ("sweeping the floor", "in a small shop"),
                 ("planting a sapling", "in a garden"), ("surfing a large wave", "in the ocean"),
                 ("skateboarding down a ramp", "at a skate park"), ("playing chess", None),
                 ("tying their shoelaces", None), ("doing a yoga pose", "on a beach at sunrise"),
                 ("fixing a bicycle", "in a garage"), ("kneading bread dough", "in a bakery"),
                 ("singing into a microphone", "on a small stage"), ("walking a tightrope", "in a circus tent"),
                 ("building a sandcastle", "on the beach"), ("shoveling snow", "in front of a house"),
                 ("writing on a chalkboard", "in a classroom"), ("running a marathon", "on a city road")]
GENERIC_PLACES = ["in a park", "in a cozy living room", "on a city sidewalk", "in a library", "in a sunny courtyard",
                  "on a train platform", "in a café", "on a balcony"]
ANIMAL_ACTS = ["sleeping curled up", "leaping through the air", "running across the grass", "drinking from a stream",
               "looking up at the sky", "playing in the snow", "yawning widely", "eating", "splashing in a puddle",
               "sitting perfectly still", "stretching", "rolling on its back"]
INTERACTIONS = ["{p1} handing {obj} to {p2}", "{p1} teaching {p2} how to ride a bicycle", "{p1} feeding {an}",
                "{p1} petting {an}", "{p1} and {p2} dancing together", "{p1} giving {p2} a high five",
                "{an} chasing {an2}", "{p1} photographing {an}", "{p1} pouring coffee for {p2}",
                "{p1} and {p2} playing tug of war with a rope", "{p1} helping {p2} climb over a fence",
                "{p1} reading a story to {p2}", "{p1} and {an} sharing an umbrella"]

def g_actions(rng, d):
    """Human actions, animal actions, interactions, simultaneous multi-agent actions."""
    kind = rng.choice(["human", "animal"]) if d <= 2 else ("interaction" if d <= 4 else "multi_agent")
    p1 = rng.choice(PEOPLE) if rng.random() < 0.6 else person(rng)
    if kind == "human":
        act, loc = rng.choice(HUMAN_ACTIONS)
        return dict(core=f"{p1} {act} {loc or rng.choice(GENERIC_PLACES)}", facet="human_action",
                    keywords=[act.split()[0]])
    if kind == "animal":
        an, act = rng.choice(LAND_ANIMALS), rng.choice(ANIMAL_ACTS)
        return dict(core=f"{a(an)} {act} {rng.choice(ANIMAL_SETTINGS)}", facet="animal_action",
                    keywords=[act.split()[0]])
    if kind == "interaction":
        p2 = rng.choice([p for p in PEOPLE if p != p1])
        an, an2 = rng.sample(LAND_ANIMALS, 2)
        t = rng.choice(INTERACTIONS)
        core = t.format(p1=p1, p2=p2, an=a(an), an2=a(an2), obj=a(rng.choice(OBJECTS)))
        return dict(core=f"{core} {rng.choice(GENERIC_PLACES)}", facet="interaction", keywords=[])
    p2 = rng.choice([p for p in PEOPLE if p != p1])
    (a1, _), (a2, _) = rng.sample(HUMAN_ACTIONS, 2)
    return dict(core=f"{p1} {a1} while {p2} is {a2} nearby, {rng.choice(GENERIC_PLACES)}", facet="multi_agent",
                keywords=[a1.split()[0], a2.split()[0]])

# ---------------- alignment / spatial relations ----------------
SPATIAL = {"left_of": "{A} to the left of {B}", "right_of": "{A} to the right of {B}",
           "above": "{A} hovering above {B}", "below": "{A} directly below {B}", "on_top_of": "{A} on top of {B}",
           "under": "{A} underneath {B}", "behind": "{A} partially hidden behind {B}",
           "in_front_of": "{A} in front of {B}", "next_to": "{A} right next to {B}", "inside": "{A} inside {C}",
           "between": "{A} between {B} and {B2}"}
SPATIAL_KW = {"left_of": "left", "right_of": "right", "above": "above", "below": "below", "on_top_of": "top",
              "under": "underneath", "behind": "behind", "in_front_of": "front", "next_to": "next",
              "inside": "inside", "between": "between"}
REL_PHRASE = {"left_of": "to the left of", "right_of": "to the right of", "behind": "behind",
              "in_front_of": "in front of", "next_to": "next to", "on_top_of": "on top of", "under": "underneath"}

def g_spatial(rng, d):
    """Single relations (d<=3), two chained relations (d=4), nested relations like inside_left_of (d=5)."""
    def np_():
        x = rng.choice(OBJECTS + ANIMALS)
        return a(f"{color(rng, d)} {x}" if d >= 3 and rng.random() < 0.7 else x)
    A, B, B2, D = np_(), np_(), np_(), np_()
    C = rng.choice(CONTAINERS)
    if d <= 3:
        rel = rng.choice(list(SPATIAL))
        return dict(core=SPATIAL[rel].format(A=A, B=B, B2=B2, C=C), facet=rel, keywords=[SPATIAL_KW[rel]])
    if d == 4:
        r1, r2 = rng.sample([r for r in SPATIAL if r not in ("inside", "between")], 2)
        core = f"{SPATIAL[r1].format(A=A, B=B)}, and {SPATIAL[r2].format(A=D, B=B)}"
        return dict(core=core, facet=f"{r1}+{r2}", keywords=[SPATIAL_KW[r1], SPATIAL_KW[r2]])
    r2 = rng.choice(list(REL_PHRASE))
    core = f"{A} inside {C}, which is {REL_PHRASE[r2]} {B}"
    if rng.random() < 0.5:
        core += f", with {D} {rng.choice(['on the far right', 'on the far left', 'in the background'])}"
    return dict(core=core, facet=f"inside_{r2}", keywords=["inside", SPATIAL_KW[r2]])

# ---------------- alignment / composition & layout ----------------
SPLIT_SCENES = [("a sunny beach", "a snowy mountain"), ("a busy city at night", "a quiet forest at dawn"),
                ("a summer meadow", "the same meadow in winter"), ("a modern kitchen", "a medieval kitchen"),
                ("a calm sea", "a stormy sea"), ("a young oak tree", "the same oak tree fully grown")]

def g_layout(rng, d):
    """Explicit placement in the frame: thirds, corners, halves, grids, circles, split screens, triptychs."""
    A, B = a(rng.choice(OBJECTS + ANIMALS)), a(rng.choice(OBJECTS + ANIMALS))
    obj = rng.choice(OBJECTS)
    options = ["rule_of_thirds", "centered_symmetric", "foreground_background"]
    if d >= 2: options += ["corner_placement", "top_bottom"]
    if d >= 3: options += ["diagonal", "grid"]
    if d >= 4: options += ["circle", "split_frame"]
    if d >= 5: options = ["circle", "split_frame", "triptych", "grid"]
    f = rng.choice(options)
    if f == "rule_of_thirds":
        side = rng.choice(["left", "right"])
        bg = rng.choice(["sky", "sand", "wall", "snowfield", "water", "fog"])
        return dict(core=f"{A} positioned on the {side} third of the frame, with empty {bg} filling the rest",
                    facet=f, keywords=[side, "third"])
    if f == "centered_symmetric":
        return dict(core=f"{A} perfectly centered in a symmetrical composition", facet=f, keywords=["center"])
    if f == "foreground_background":
        return dict(core=f"{A} in sharp focus in the foreground and {B} blurred in the background", facet=f,
                    keywords=["foreground", "background"])
    if f == "corner_placement":
        c1, c2 = rng.choice([("top-left", "bottom-right"), ("top-right", "bottom-left")])
        return dict(core=f"{A} in the {c1} corner and {B} in the {c2} corner of the image, the rest of the frame "
                         f"empty", facet=f, keywords=[c1, c2])
    if f == "top_bottom":
        return dict(core=f"{A} in the upper half of the image and {B} in the lower half", facet=f,
                    keywords=["upper", "lower"])
    if f == "diagonal":
        n = rng.randint(3, 6)
        return dict(core=f"{NUM_WORDS[n]} {plural(obj)} arranged along a diagonal line from the bottom-left to "
                         f"the top-right", facet=f, keywords=[NUM_WORDS[n], "diagonal"])
    if f == "grid":
        r, c = rng.randint(2, 3 if d < 5 else 4), rng.randint(2, 4)
        return dict(core=f"a {r}x{c} grid of {plural(obj)}, each one a different color", facet=f,
                    keywords=[f"{r}x{c}"])
    if f == "circle":
        n = rng.randint(4, 8)
        return dict(core=f"{NUM_WORDS[n]} {plural(obj)} arranged in a circle around {B}", facet=f,
                    keywords=[NUM_WORDS[n], "circle"])
    if f == "split_frame":
        x, y = rng.choice(SPLIT_SCENES)
        return dict(core=f"a split-screen image: the left half shows {x} and the right half shows {y}", facet=f,
                    keywords=["left", "right"])
    x, y, z = rng.sample([s for pair in SPLIT_SCENES for s in pair], 3)
    return dict(core=f"three vertical panels side by side showing {x}, {y} and {z}", facet="triptych",
                keywords=["three"])

print("alignment generators ready")

alignment generators ready


## 6. Generators — Visual quality

A generic factory `make_vq` builds generators from `(subject, facet)` lists plus sub-dimension specific shot types,
quality descriptors and lighting. Difficulty adds shot type → quality requirement → lighting → (for some)
a second, contrasting subject. `skin` has a dedicated generator for demographic diversity.

In [25]:
# ---------------- visual quality: generic factory ----------------
def make_vq(subjects, shots, quality, lights, combo=True):
    """Build a generator for a visual-quality sub-dimension."""
    def g(rng, d):
        s, facet = rng.choice(subjects)
        extras = []
        if d >= 2: extras.append(rng.choice(shots))
        if d >= 3: extras.append(rng.choice(quality))
        if d >= 4: extras.append(rng.choice(lights))
        if d == 5 and combo:
            s2, f2 = rng.choice([x for x in subjects if x[1] != facet])
            return dict(core=f"{s} beside {s2}", facet=f"{facet}+{f2}", keywords=[], extras=extras)
        if d == 5:
            extras.append(rng.choice([q for q in quality if q not in extras]))
        return dict(core=s, facet=facet, keywords=[], extras=extras)
    return g

TEXTURES = [("weathered tree bark", "organic"), ("a knitted wool sweater", "fabric"), ("cracked desert earth", "mineral"),
            ("rusty corrugated metal", "weathered"), ("a woven rattan basket", "fabric"),
            ("peeling paint on an old wooden door", "weathered"), ("a moss-covered stone", "organic"),
            ("coarse sea salt crystals", "mineral"), ("a burlap sack", "fabric"), ("a slice of sourdough bread", "food"),
            ("a worn leather saddle", "weathered"), ("a crumpled sheet of paper", "paper"), ("a honeycomb", "organic"),
            ("a stack of old books", "paper"), ("an orange peel", "food"), ("a hand-thrown clay pot", "mineral"),
            ("a faded denim jacket", "fabric"), ("a pineapple", "food"), ("wet sand with footprints", "mineral"),
            ("a velvet cushion", "fabric"), ("lichen on a granite boulder", "organic"), ("a croissant", "food")]
TRANSPARENT = [("a crystal wine glass filled with red wine", "glass"), ("a glass teapot with blooming tea", "glass"),
               ("ice cubes in a glass of water", "ice"), ("a soap bubble reflecting a rainbow", "bubble"),
               ("a jellyfish drifting in dark water", "organic"), ("a faceted perfume bottle", "glass"),
               ("raindrops on a window pane", "liquid"), ("a glass chess set", "glass"),
               ("a transparent umbrella in the rain", "plastic"), ("a flower frozen inside a block of ice", "ice"),
               ("a prism splitting light into a rainbow", "caustics"), ("a glass of sparkling water", "liquid"),
               ("a quartz crystal cluster", "crystal"), ("a translucent leaf backlit by the sun", "translucent"),
               ("a glass sculpture of a horse", "glass"), ("water splashing out of a glass", "liquid"),
               ("a snow globe with a tiny village", "glass"), ("an aquarium with tropical fish", "liquid"),
               ("a clear glass marble", "glass"), ("honey dripping from a glass jar", "liquid")]
REFLECTIVE = [("a chrome motorcycle", "metal"), ("a polished silver teapot", "metal"),
              ("an ornate mirror reflecting a candlelit room", "mirror"), ("a wet city street reflecting neon lights",
              "wet_surface"), ("a calm mountain lake reflecting the peaks", "water"),
              ("a chrome sphere on a checkerboard floor", "metal"), ("aviator sunglasses reflecting a beach", "mirror"),
              ("a glossy black grand piano", "glossy"), ("a polished marble floor in a palace hall", "glossy"),
              ("a copper kettle", "metal"), ("a knight's polished steel armor", "metal"),
              ("a skyscraper with a mirrored glass facade", "mirror"), ("a puddle reflecting a red umbrella",
              "wet_surface"), ("a golden trophy", "metal"), ("a freshly waxed sports car", "glossy"),
              ("a spoon reflecting a face upside down", "metal"), ("a rice paddy mirroring the sky", "water")]
FUR = [("a Persian cat", "long_fur"), ("a husky", "double_coat"), ("a highland cow", "shaggy"), ("a red fox", "dense"),
       ("an alpaca", "wool"), ("a golden retriever", "long_fur"), ("a male lion with a full mane", "mane"),
       ("a snowy owl", "feathers"), ("a chinchilla", "dense"), ("a hamster", "short"), ("an old English sheepdog",
       "shaggy"), ("a baby rabbit", "soft"), ("a peacock displaying its tail", "feathers"), ("a polar bear", "dense"),
       ("a Maine Coon cat", "long_fur"), ("a red squirrel", "bushy_tail"), ("a yak", "shaggy"),
       ("an arctic fox in its winter coat", "dense"), ("a wet otter", "wet_fur"), ("a fluffy duckling", "down")]
FOLIAGE = [("a dense tropical rainforest canopy", "rainforest"), ("a Japanese maple in autumn", "tree"),
           ("ferns unfurling on a forest floor", "ferns"), ("a field of tall grass swaying in the wind", "grass"),
           ("a bonsai tree", "tree"), ("ivy covering a brick wall", "vines"), ("a lavender field", "flowers"),
           ("cherry blossom branches", "flowers"), ("a pine forest after snowfall", "forest"),
           ("a moss-covered forest floor", "moss"), ("a wildflower meadow", "flowers"),
           ("a willow tree by a pond", "tree"), ("dew drops on leaves at dawn", "detail"), ("a bamboo grove", "bamboo"),
           ("a lush vegetable garden", "garden"), ("an old oak tree in a field", "tree"),
           ("autumn leaves covering a path", "leaves"), ("a hedge maze seen from above", "hedge")]
LOW_LIGHT = [("a quiet room lit only by a single candle", "candlelight"), ("a village under a full moon", "moonlight"),
             ("a rainy alley lit by neon signs", "neon"), ("friends around a campfire in the woods", "campfire"),
             ("bioluminescent waves on a beach at night", "bioluminescence"), ("the Milky Way over a desert", "starlight"),
             ("a city skyline at night", "city_night"), ("a paper lantern festival at night", "lantern"),
             ("a harbor during blue hour", "blue_hour"), ("a lone figure under a streetlight in fog", "streetlight"),
             ("a face lit by a laptop screen in a dark room", "screen_glow"), ("fireflies in a forest at dusk",
             "fireflies"), ("the aurora borealis over a frozen lake", "aurora"), ("a dimly lit subway tunnel",
             "underground"), ("a jazz singer under a single spotlight", "stage"), ("a night train interior", "interior")]
FINE_DETAIL = [("the intricate gears of an open pocket watch", "mechanical"), ("a lace wedding veil", "textile"),
               ("a detailed circuit board", "electronics"), ("an ornate Persian carpet", "pattern"),
               ("a dragonfly wing", "nature"), ("a filigree silver necklace", "jewelry"),
               ("an aerial view of a dense old city", "aerial"), ("a gothic cathedral facade with stone carvings",
               "architecture"), ("a snowflake", "nature"), ("a hand-embroidered silk kimono", "textile"),
               ("a stained glass rose window", "pattern"), ("a ship in a bottle", "miniature"),
               ("a miniature model railway town", "miniature"), ("the scales of a python", "nature"),
               ("a luthier's workbench full of tools", "workspace"), ("a crowded antique bookshelf", "clutter"),
               ("a mandala drawn in fine ink", "pattern"), ("a peacock feather eye", "nature"),
               ("a mechanical typewriter", "mechanical"), ("a coral reef teeming with small fish", "nature")]

VQ_SHOTS_MACRO = ["macro shot", "extreme close-up", "close-up view", "detailed close-up"]
VQ_SHOTS_WIDE = ["wide shot", "medium shot", "eye-level view", "close-up view"]
g_textures = make_vq(TEXTURES, VQ_SHOTS_MACRO, ["every grain and fiber visible", "tactile surface detail",
                     "ultra-detailed texture"], ["raking side light revealing the texture", "soft diffuse light"])
g_transparent = make_vq(TRANSPARENT, VQ_SHOTS_MACRO + ["product shot"], ["realistic refraction",
                        "accurate caustics", "crystal-clear transparency"], ["backlit", "bright window light",
                        "dark background with rim light"])
g_reflective = make_vq(REFLECTIVE, VQ_SHOTS_WIDE + ["product shot"], ["accurate reflections",
                       "mirror-like surface", "clean specular highlights"], ["studio lighting with softboxes",
                       "sunset light", "night lighting"])
g_fur = make_vq(FUR, ["close-up portrait", "full-body shot", "medium shot"], ["individual hairs visible",
                "soft fluffy fur detail", "fur ruffled by the wind"], ["golden hour backlight", "soft overcast light",
                "morning light"], combo=False)
g_foliage = make_vq(FOLIAGE, VQ_SHOTS_WIDE + ["aerial view"], ["every leaf rendered sharply",
                    "lush dense vegetation", "light filtering through the leaves"], ["dappled sunlight",
                    "misty morning light", "late afternoon light"], combo=False)
g_low_light = make_vq(LOW_LIGHT, ["wide shot", "medium shot", "long exposure"], ["deep shadows with clean detail",
                      "low noise night photography", "high dynamic range at night"], ["only practical light sources",
                      "strong contrast between light and shadow"], combo=False)
g_fine_detail = make_vq(FINE_DETAIL, VQ_SHOTS_MACRO + ["top-down view"], ["extremely intricate detail",
                        "razor-sharp fine detail", "microscopic precision"], ["even studio lighting",
                        "soft window light"])

# ---------------- visual quality: skin ----------------
SKIN_DETAILS = [("with freckles across the nose", "freckles"), ("with deep laugh lines and wrinkles", "wrinkles"),
                ("with a light sheen of sweat after a run", "sweat"), ("with vitiligo patches", "vitiligo"),
                ("with natural bare skin and no makeup", "natural"), ("with a few moles and visible pores", "pores"),
                ("with sun-weathered skin", "weathered"), ("with a fine-line tattoo on the neck", "tattoo"),
                ("with rosy cheeks in the cold", "flush"), ("with water droplets on the face", "wet"),
                ("with albinism and pale eyelashes", "albinism"), ("with a scar across the eyebrow", "scar")]
SKIN_HANDS = ["the weathered hands of an old potter shaping clay", "a baby's hand holding a grandparent's finger",
              "the calloused hands of a guitarist", "hands kneading bread dough dusted with flour"]

def g_skin(rng, d):
    """Realistic skin: diverse ages/ethnicities, skin details, hands."""
    extras = []
    if rng.random() < 0.15:
        core, facet = rng.choice(SKIN_HANDS), "hands"
    else:
        detail, facet = rng.choice(SKIN_DETAILS)
        core = f"{person(rng)} {detail}"
    if d >= 2: extras.append(rng.choice(["close-up portrait", "headshot", "extreme close-up of the face"]))
    if d >= 3: extras.append(rng.choice(["natural skin texture with visible pores", "realistic subsurface scattering",
                                         "no retouching"]))
    if d >= 4: extras.append(rng.choice(["soft window light", "harsh midday sun", "rim light", "overcast light"]))
    if d >= 5: extras.append(rng.choice(["85mm lens", "shallow depth of field"]))
    return dict(core=core, facet=facet, keywords=[], extras=extras)

print("visual quality generators ready")

visual quality generators ready


## 7. Generators — Aesthetics

Composition techniques, lighting setups, color harmony, portraits, landscapes, cinematic framing and illustration
styles. The aesthetic property is carried by mandatory `extras` (e.g. `"chiaroscuro lighting"`) so that it is kept
whatever the prompting format.

In [26]:
# ---------------- aesthetics / composition ----------------
COMPOSITIONS = {
    "leading_lines": ["{s} at the end of a long road stretching into the distance",
                      "railway tracks leading the eye towards {s}", "a winding river guiding the eye to {s}"],
    "negative_space": ["{s} small in the frame, surrounded by vast empty {space}"],
    "natural_frame": ["{s} framed by {frame}"],
    "symmetry": ["{s} at the center of a perfectly symmetrical {sym}"],
    "rule_of_thirds": ["{s} placed on a rule-of-thirds intersection, {land} in the background"],
    "layering": ["{s} in the foreground, layers of misty hills receding into the distance"],
    "pattern_break": ["rows of identical white umbrellas with a single {c} one", "a field of identical {c2} tulips "
                      "with one {c} tulip in the middle"],
    "minimalism": ["{s} alone against a plain {c} background, minimalist composition"],
    "golden_spiral": ["{s} placed at the eye of a golden spiral composition"],
}
SPACE = ["sky", "sea", "snowfield", "desert", "fog", "salt flat"]
FRAMES = ["an old stone archway", "a window", "overhanging tree branches", "a cave entrance", "a doorway",
          "a torii gate", "a circular moon gate"]
SYMS = ["corridor", "hallway of arches", "bridge", "formal garden", "staircase"]
LANDS = ["rolling hills", "a mountain range", "the sea", "a city skyline", "a pine forest"]

def g_composition(rng, d):
    """Classical composition techniques."""
    facet = rng.choice(list(COMPOSITIONS))
    core = rng.choice(COMPOSITIONS[facet]).format(
        s=rng.choice(GENERAL_SUBJECTS), space=rng.choice(SPACE), frame=rng.choice(FRAMES), sym=rng.choice(SYMS),
        land=rng.choice(LANDS), c=rng.choice(COLOR_BASIC), c2=rng.choice(["white", "yellow", "pink"]))
    extras = []
    if d >= 3: extras.append(rng.choice(MOOD_OPT))
    if d >= 4: extras.append(rng.choice(LIGHTING_OPT))
    if d >= 2: extras.insert(0, rng.choice(["balanced composition", "strong visual hierarchy", "elegant composition"]))
    return dict(core=core, facet=facet, keywords=[], extras=extras)

# ---------------- aesthetics / lighting ----------------
LIGHTS = {"golden_hour": "warm golden hour light", "blue_hour": "cool blue hour light",
          "rim_light": "strong rim lighting outlining the subject", "chiaroscuro": "chiaroscuro lighting",
          "softbox": "soft even softbox lighting", "volumetric": "volumetric light rays through haze",
          "backlit": "backlit by the setting sun", "overcast": "soft overcast light",
          "harsh_noon": "harsh midday sunlight with hard shadows", "neon": "colorful neon lighting",
          "candle": "warm flickering candlelight", "split_light": "split lighting, half the face in shadow",
          "rembrandt": "Rembrandt lighting", "dappled": "dappled light through leaves",
          "silhouette": "the subject in silhouette against a bright sky"}

def g_lighting(rng, d):
    """Named lighting setups applied to generic scenes."""
    facet = rng.choice(list(LIGHTS))
    core = rng.choice(GENERAL_SUBJECTS + PEOPLE) if d <= 2 else rng.choice(SCENES)
    extras = [LIGHTS[facet]]
    if d >= 4: extras.append(rng.choice(["long shadows", "glowing highlights", "atmospheric haze", "deep contrast"]))
    if d >= 5: extras.append(rng.choice(MOOD_OPT))
    return dict(core=core, facet=facet, keywords=[], extras=extras)

# ---------------- aesthetics / color harmony ----------------
HARMONIES = {"complementary": "a complementary orange and teal color palette",
             "analogous": "an analogous palette of blues and greens", "monochrome": "a monochromatic palette of {c}s",
             "pastel": "soft pastel colors", "triadic": "a triadic palette of red, yellow and blue",
             "earth_tones": "warm earth tones", "black_and_white": "black and white, high contrast",
             "neon_palette": "a vivid neon palette of pink and cyan", "muted": "muted desaturated tones",
             "sepia": "sepia tones", "jewel_tones": "rich jewel tones of emerald, sapphire and ruby",
             "warm_cool": "warm foreground colors against a cool background"}

def g_color_harmony(rng, d):
    """Global color palettes."""
    facet = rng.choice(list(HARMONIES))
    palette = HARMONIES[facet].format(c=rng.choice(["red", "blue", "green", "yellow", "purple"]))
    core = rng.choice(SCENES)
    extras = [palette] + ([rng.choice(["harmonious colors", "cohesive color grading"])] if d >= 3 else [])
    if d >= 4: extras.append(rng.choice(LIGHTING_OPT))
    return dict(core=core, facet=facet, keywords=[], extras=extras)

# ---------------- aesthetics / portraits ----------------
PORTRAIT_SHOTS = {"headshot": "close-up headshot", "half_body": "half-body portrait", "full_body": "full-body portrait",
                  "environmental": "environmental portrait", "profile": "profile view portrait"}
EXPRESSIONS = ["smiling warmly", "laughing", "looking thoughtful", "with a serious expression",
               "looking over the shoulder", "gazing out of a window", "with eyes closed", "with a confident smile"]
PORTRAIT_PLACES = ["in a sunlit studio", "in a busy street", "in a flower garden", "against a plain gray backdrop",
                   "in a workshop", "by the sea", "in a library", "in a wheat field", "in a neon-lit bar"]

def g_portraits(rng, d):
    """Portraits: shot types, group portraits, expressions, places."""
    if d >= 4 and rng.random() < 0.35:
        n = rng.randint(2, 5)
        who = rng.choice(["friends", "family members", "colleagues", "musicians", "grandparents and grandchildren"])
        return dict(core=f"{NUM_WORDS[n]} {who} posing together {rng.choice(PORTRAIT_PLACES)}", facet="group",
                    keywords=[NUM_WORDS[n]], extras=["group portrait", rng.choice(LIGHTING_OPT)])
    facet = rng.choice(list(PORTRAIT_SHOTS))
    who = rng.choice(PEOPLE) if rng.random() < 0.5 else person(rng)
    core = f"{who} {rng.choice(EXPRESSIONS)}" + (f" {rng.choice(PORTRAIT_PLACES)}" if d >= 2 else "")
    extras = [PORTRAIT_SHOTS[facet]] + ([rng.choice(LIGHTING_OPT)] if d >= 3 else [])
    if d >= 5: extras.append(rng.choice(["wearing a hand-knitted sweater", "holding a cup of coffee",
                                         "with wind in their hair"]))
    return dict(core=core, facet=facet, keywords=[], extras=extras)

# ---------------- aesthetics / landscapes ----------------
TERRAINS = {"mountains": "a jagged snow-capped mountain range", "coast": "a rugged coastline with sea stacks",
            "desert": "rolling sand dunes", "forest": "an ancient redwood forest", "lake": "a glassy alpine lake",
            "canyon": "a deep red-rock canyon", "tundra": "a vast arctic tundra",
            "rice_terraces": "terraced rice fields on a hillside", "volcanic": "a smoking volcano over black lava fields",
            "waterfall": "a tall waterfall in a lush gorge", "savanna": "an acacia-dotted savanna",
            "fjord": "a steep-walled fjord", "karst": "karst limestone peaks along a river",
            "meadow": "an alpine flower meadow", "glacier": "a blue glacier tongue", "salt_flat": "a mirror-like salt flat"}
TIMES = ["at sunrise", "under storm clouds", "in thick fog", "at golden hour", "under a starry sky",
         "after fresh snowfall", "in autumn colors", "with a rainbow after rain", "at dusk", "on a clear summer day"]
FOREGROUNDS = ["with a lone hiker in the foreground", "with wildflowers in the foreground",
               "with a winding path in the foreground", "with a small red cabin in the foreground",
               "with grazing horses in the foreground"]

def g_landscapes(rng, d):
    """Landscapes: terrain x time/weather x foreground element."""
    facet = rng.choice(list(TERRAINS))
    core = TERRAINS[facet] + (f" {rng.choice(TIMES)}" if d >= 2 else "")
    if d >= 3: core += f" {rng.choice(FOREGROUNDS)}"
    extras = [rng.choice(["wide panoramic view", "sweeping vista", "aerial view"])] if d >= 4 else []
    return dict(core=core, facet=facet, keywords=[], extras=extras)

# ---------------- aesthetics / cinematic framing ----------------
CINE_SHOTS = {"establishing_wide": "wide establishing shot", "over_the_shoulder": "over-the-shoulder shot",
              "low_angle": "low-angle shot", "high_angle": "high-angle shot", "dutch_angle": "dutch angle",
              "extreme_closeup": "extreme close-up", "tracking": "tracking shot with motion blur",
              "silhouette": "silhouetted figure against the light", "anamorphic": "anamorphic widescreen framing",
              "birds_eye": "bird's-eye view", "two_shot": "two-shot framing", "deep_focus": "deep focus composition"}
CINE_SCENES = ["a detective walking down a rain-soaked alley", "a lone astronaut on a red planet",
               "a samurai standing in a bamboo forest", "a family dinner in a 1970s kitchen",
               "a car speeding along a desert highway", "two lovers saying goodbye on a train platform",
               "a knight facing a dragon at the mouth of a cave", "a scientist in an abandoned laboratory",
               "a cowboy pushing open saloon doors", "a girl riding a bicycle through a small coastal town",
               "a submarine crew in a red-lit control room", "a chess match in a smoky club",
               "a boxer resting in the corner of the ring", "an old fisherman mending nets at dawn"]

def g_cinematic(rng, d):
    """Cinematic shot types applied to narrative scenes."""
    facet = rng.choice(list(CINE_SHOTS))
    extras = [CINE_SHOTS[facet]]
    if d >= 3: extras.append(rng.choice(["teal and orange color grade", "moody color grading", "film grain",
                                         "high contrast"]))
    if d >= 4: extras.append(rng.choice(["volumetric fog", "practical lights", "lens flare", "motion blur"]))
    return dict(core=rng.choice(CINE_SCENES), facet=facet, keywords=[], extras=extras)

# ---------------- aesthetics / illustration styles ----------------
ILLUSTRATION_STYLE_KEYS = [k for k in STYLES if k not in ("photographic", "cinematic", "graphic_design")]

def g_illustration(rng, d):
    """The same kind of subjects rendered in many illustration media (style forced = facet)."""
    style = rng.choice(ILLUSTRATION_STYLE_KEYS)
    core = rng.choice(GENERAL_SUBJECTS) if d == 1 else rng.choice(SCENES)
    extras = [rng.choice(MOOD_OPT)] if d >= 3 else []
    return dict(core=core, facet=style, keywords=[], extras=extras, style=style)

print("aesthetics generators ready")

aesthetics generators ready


## 8. Generators — Real-world fidelity

Architecture, interiors, vehicles, clothing, everyday objects, culturally specific scenes and physically plausible
interactions. Cultural scenes are written descriptively and respectfully, covering many regions.

In [27]:
# ---------------- real world / architecture ----------------
ARCH_STYLES = ["Gothic", "brutalist", "Art Deco", "traditional Japanese", "Moorish", "Scandinavian modern",
               "Victorian", "Bauhaus", "Mughal", "half-timbered Bavarian", "whitewashed Mediterranean",
               "Chinese imperial", "adobe Pueblo", "Beaux-Arts", "Russian Orthodox", "Khmer", "Ottoman",
               "Tudor", "deconstructivist", "Brazilian modernist"]
BUILDINGS = ["cathedral", "apartment block", "train station", "family house", "library", "museum", "bridge",
             "tower", "temple", "school", "market hall", "town hall", "tea house", "lighthouse", "hotel"]
ARCH_VIEWS = ["exterior view", "street-level view", "aerial view", "view from across a plaza", "facade detail"]

def g_architecture(rng, d):
    """Architectural styles x building types."""
    style = rng.choice(ARCH_STYLES)
    core = a(f"{style} {rng.choice(BUILDINGS)}")
    if d >= 3: core += f" {rng.choice(['at dusk', 'in the rain', 'on a sunny morning', 'in winter', 'at night'])}"
    if d >= 4: core += f", {rng.choice(['with people walking by', 'with cars parked in front', 'with a small park'])}"
    extras = [rng.choice(ARCH_VIEWS)] if d >= 2 else []
    if d >= 5: extras.append("architecturally accurate proportions")
    return dict(core=core, facet=re.sub(r"\W+", "_", style.lower()), keywords=[], extras=extras)

# ---------------- real world / interiors ----------------
ROOMS = ["kitchen", "living room", "bedroom", "bathroom", "home office", "library", "café", "restaurant",
         "hotel lobby", "classroom", "hospital corridor", "recording studio", "woodworking workshop", "laundromat",
         "train carriage", "attic", "greenhouse", "bakery"]
DESIGNS = ["mid-century modern", "Japandi", "industrial", "farmhouse", "maximalist", "minimalist", "Moroccan",
           "Art Deco", "Scandinavian", "bohemian", "Victorian", "traditional Korean hanok", "1980s retro"]
INTERIOR_DETAILS = ["with plants on the windowsill", "with a cat asleep on the sofa", "with books stacked everywhere",
                    "with morning light across the floor", "with a half-finished cup of tea on the table",
                    "with rain on the windows"]

def g_interiors(rng, d):
    """Rooms x interior design styles x lived-in details."""
    room = rng.choice(ROOMS)
    core = a(f"{rng.choice(DESIGNS)} {room}")
    if d >= 3: core += f" {rng.choice(INTERIOR_DETAILS)}"
    extras = [rng.choice(["wide-angle interior shot", "eye-level view", "corner view"])] if d >= 2 else []
    if d >= 4: extras.append(rng.choice(["realistic materials", "accurate perspective", "natural light"]))
    return dict(core=core, facet=re.sub(r"\W+", "_", room), keywords=[], extras=extras)

# ---------------- real world / vehicles ----------------
VEHICLES = [("a vintage convertible", "car"), ("an electric scooter", "two_wheel"), ("a steam locomotive", "rail"),
            ("a container ship", "water"), ("a fighter jet", "air"), ("a hot air balloon", "air"),
            ("a red double-decker bus", "road_public"), ("a tuk-tuk", "road_public"), ("a sailboat", "water"),
            ("a cafe racer motorcycle", "two_wheel"), ("a pickup truck", "car"), ("a bullet train", "rail"),
            ("a tram", "rail"), ("a cargo bicycle", "two_wheel"), ("a tractor", "utility"),
            ("a rescue helicopter", "air"), ("a canoe", "water"), ("a fire truck", "utility"),
            ("a cycle rickshaw", "road_public"), ("a snowmobile", "utility"), ("a yellow taxi", "car"),
            ("a camper van", "car"), ("a gondola", "water"), ("a vintage biplane", "air")]
VEHICLE_SETTINGS = ["on a mountain road", "in a busy city intersection", "parked by the sea", "in a snowy landscape",
                    "at a rural station", "in a garage", "crossing a bridge", "in a desert"]

def g_vehicles(rng, d):
    """Vehicles in context, with correct mechanical details at higher difficulty."""
    v, facet = rng.choice(VEHICLES)
    core = v + (f" {rng.choice(VEHICLE_SETTINGS)}" if d >= 2 else "")
    extras = []
    if d >= 3: extras.append(rng.choice(["three-quarter view", "side view", "front view", "motion blur background"]))
    if d >= 4: extras.append(rng.choice(["mechanically accurate details", "realistic wheels and proportions",
                                         "visible wear and dirt"]))
    return dict(core=core, facet=facet, keywords=[], extras=extras)

# ---------------- real world / clothing ----------------
GARMENTS = [("a tailored wool overcoat", "outerwear"), ("a silk qipao", "traditional"),
            ("a denim jacket with embroidered patches", "casual"), ("a hanbok", "traditional"),
            ("a sari with gold borders", "traditional"), ("a kimono with a crane pattern", "traditional"),
            ("a leather biker jacket", "outerwear"), ("a chunky knitted cardigan", "casual"),
            ("a tartan kilt", "traditional"), ("a dashiki", "traditional"), ("a black tuxedo", "formal"),
            ("a yellow raincoat", "outerwear"), ("a ball gown", "formal"), ("technical hiking gear", "sport"),
            ("a football kit", "sport"), ("a chef's uniform", "work"), ("a flamenco dress", "traditional"),
            ("a woven poncho", "traditional"), ("an oversized hoodie", "casual"), ("a trench coat", "outerwear"),
            ("a welder's protective gear", "work"), ("a linen summer suit", "formal")]
POSES = ["standing", "walking towards the camera", "sitting on steps", "turning around", "leaning against a wall"]

def g_clothing(rng, d):
    """Garments (fabric, fit, cultural attire) worn by diverse people."""
    g, facet = rng.choice(GARMENTS)
    who = rng.choice(PEOPLE) if rng.random() < 0.4 else person(rng)
    core = f"{who} wearing {g}" + (f", {rng.choice(POSES)}" if d >= 2 else "")
    if d >= 3: core += f" {rng.choice(PORTRAIT_PLACES)}"
    extras = [rng.choice(["fashion photography", "full-body shot", "street style photo"])] if d >= 2 else []
    if d >= 4: extras.append(rng.choice(["detailed fabric texture", "accurate stitching and seams",
                                         "natural fabric drape"]))
    return dict(core=core, facet=facet, keywords=[], extras=extras)

# ---------------- real world / everyday objects ----------------
EVERYDAY = ["toaster", "bicycle helmet", "smartphone", "pair of headphones", "alarm clock", "electric kettle",
            "stapler", "pair of sneakers", "backpack", "umbrella", "desk lamp", "TV remote", "toothbrush",
            "frying pan", "laptop", "wristwatch", "pair of reading glasses", "coffee grinder", "houseplant",
            "set of keys", "water bottle", "wallet", "hairdryer", "mechanical keyboard", "sewing machine"]

def g_everyday(rng, d):
    """Everyday objects: product shots, in-context use, flat lays, cluttered surfaces."""
    facet = rng.choice(["product", "in_context"] if d <= 2 else ["product", "in_context", "flat_lay", "cluttered"])
    o = rng.sample(EVERYDAY, 4)
    if facet == "product":
        return dict(core=a(o[0]) + f" on {rng.choice(['a white background', 'a wooden surface', 'a pastel background'])}",
                    facet=facet, keywords=[], extras=["product photo"] if d >= 2 else [])
    if facet == "in_context":
        return dict(core=f"{a(o[0])} being used {rng.choice(GENERIC_PLACES)}", facet=facet, keywords=[],
                    extras=[rng.choice(["lifestyle photo", "candid shot"])] if d >= 2 else [])
    if facet == "flat_lay":
        return dict(core=f"{join_list([a(x) for x in o[:3]])} neatly arranged", facet=facet, keywords=[],
                    extras=["flat lay", "top-down view"])
    return dict(core=f"a messy desk with {join_list([a(x) for x in o])}", facet=facet, keywords=[],
                extras=[rng.choice(["realistic clutter", "natural light"])])

# ---------------- real world / culturally specific scenes ----------------
CULTURAL = [("families lighting oil lamps during Diwali", "south_asia"),
            ("a Lunar New Year street market with red lanterns", "east_asia"),
            ("a Día de los Muertos altar with marigolds and candles", "latin_america"),
            ("a Japanese tea ceremony in a tatami room", "east_asia"),
            ("a spice stall in a Moroccan souk", "north_africa"), ("people throwing colored powder during Holi", "south_asia"),
            ("a Nowruz table with the haft-sin arrangement", "middle_east"),
            ("a Songkran water festival street in Thailand", "southeast_asia"),
            ("a Maasai beadwork market in Kenya", "east_africa"), ("a Mongolian ger on the steppe", "central_asia"),
            ("a Scandinavian Midsummer celebration with a flower maypole", "europe"),
            ("a Korean family preparing kimchi together", "east_asia"),
            ("a floating market in Vietnam with boats full of fruit", "southeast_asia"),
            ("an Ethiopian coffee ceremony", "east_africa"), ("a Venetian carnival with ornate masks", "europe"),
            ("a Peruvian weaver working on a backstrap loom", "latin_america"),
            ("a Nigerian wedding with guests in aso-ebi fabric", "west_africa"),
            ("a Ramadan iftar meal shared on a long table", "middle_east"),
            ("a Hawaiian hula performance at sunset", "oceania"), ("a Chinese dragon boat race", "east_asia"),
            ("an Irish pub session with fiddles and bodhrán", "europe"),
            ("a Brazilian samba school parade", "latin_america"), ("a Japanese summer festival with yukata", "east_asia"),
            ("a Sámi reindeer herder in northern Norway", "europe"),
            ("a Balinese temple offering ceremony", "southeast_asia"), ("a Mexican mariachi band in a plaza", "latin_america")]

def g_cultural(rng, d):
    """Culturally specific scenes from many regions."""
    scene, facet = rng.choice(CULTURAL)
    extras = []
    if d >= 2: extras.append(rng.choice(["documentary photography", "authentic details", "candid moment"]))
    if d >= 4: extras.append(rng.choice(["accurate traditional clothing", "culturally accurate details",
                                         "respectful portrayal"]))
    return dict(core=scene, facet=facet, keywords=[], extras=extras)

# ---------------- real world / physically plausible interactions ----------------
PHYSICS = [("water being poured from a pitcher into a glass, with small splashes", "fluid"),
           ("a stack of books balanced precariously on a chair", "balance"),
           ("a basketball mid-bounce casting a shadow on the court", "shadow"),
           ("a heavy anvil sinking into a soft cushion", "deformation"), ("a white sheet draped over an armchair", "cloth"),
           ("smoke curling up from a just-extinguished candle", "smoke"), ("fresh footprints in deep snow", "trace"),
           ("a spoon in a glass of water appearing bent", "refraction"),
           ("wind blowing a woman's hair and scarf sideways", "wind"), ("a line of dominoes mid-fall", "motion"),
           ("ripples spreading from a stone thrown into a pond", "fluid"), ("a boot pressing into thick mud", "deformation"),
           ("a palm tree bending in a storm", "wind"), ("sunlight casting window-shaped shadows across a floor", "shadow"),
           ("a glass shattering on a tiled floor", "fracture"), ("ice cream melting and dripping down a cone", "melting"),
           ("a hand squeezing a soft stress ball", "deformation"), ("a cat knocking a cup off a table", "motion"),
           ("rain bouncing off a car roof", "fluid"), ("two people pulling a rope in a tug of war", "force"),
           ("a paper boat floating in a gutter stream", "buoyancy"), ("a heavy chain hanging in a curve", "gravity"),
           ("a child on a swing at the highest point of the arc", "gravity"), ("a tablecloth being pulled from a set table",
           "motion")]

def g_physics(rng, d):
    """Physical plausibility: fluids, shadows, deformation, gravity, cloth, motion."""
    core, facet = rng.choice(PHYSICS)
    extras = []
    if d >= 2: extras.append(rng.choice(["high-speed photography", "frozen motion", "realistic physics"]))
    if d >= 4: extras.append(rng.choice(["physically accurate shadows", "correct scale and weight",
                                         "consistent light direction"]))
    return dict(core=core, facet=facet, keywords=[], extras=extras)

print("real-world generators ready")

real-world generators ready


## 9. Generators — Creative generation

Concept art, character design, posters, comics, **text rendering** (English + Chinese strings, multi-text, long text),
surreal scenes, storytelling and unusual compositions. All rendered text is placed inside double quotes so it can be
validated verbatim after LLM rewriting.

In [28]:
# ---------------- creative / concept art ----------------
CONCEPT = {"environment": ["a floating city above the clouds", "an abandoned space station overgrown with plants",
                           "a desert outpost built from salvaged ships", "an underwater research base",
                           "a crystal cave with glowing minerals", "a cyberpunk megacity market",
                           "a mountain monastery carved into a cliff", "an alien jungle with towering fungi",
                           "a steampunk airship dock", "a frozen city under an ice dome"],
           "creature": ["a six-legged desert predator", "a gentle forest giant covered in moss",
                        "a bioluminescent deep-sea leviathan", "a dragon with obsidian scales",
                        "a small fox-like creature with crystal antlers", "a winged serpent",
                        "a giant armored beetle used as a mount"],
           "vehicle": ["a hover tank", "a solar-sail spaceship", "a walking farming mech", "a desert sand-skimmer",
                       "an ornithopter", "a deep-sea exploration submarine"],
           "prop": ["an ornate magic sword", "a sci-fi medical scanner", "an ancient relic with glowing runes",
                    "a steampunk pocket telescope", "a shaman's staff"]}

def g_concept(rng, d):
    """Concept art: environments, creatures, vehicles, props."""
    facet = rng.choice(list(CONCEPT))
    extras = []
    if d >= 2: extras.append(rng.choice(["epic scale", "sense of scale with tiny figures", "dramatic atmosphere"]))
    if d >= 3: extras.append(rng.choice(["detailed design", "clear silhouette", "cohesive color keys"]))
    if d >= 5 and facet != "environment": extras.append("design sheet with annotations")
    return dict(core=rng.choice(CONCEPT[facet]), facet=facet, keywords=[], extras=extras)

# ---------------- creative / character design ----------------
ARCHETYPES = ["a young wizard apprentice", "a retired space pirate", "a desert nomad courier", "a cyberpunk hacker",
              "a forest ranger with a pet owl", "a steampunk inventor", "a knight in patched armor",
              "a street food vendor in a fantasy city", "a deep-sea diver", "a samurai cat", "a robot gardener",
              "a jazz-age detective", "a mountain shepherd", "a clockwork doll", "a kind witch who runs a bakery"]
SHEETS = {"single": "character portrait", "full_body": "full-body character design",
          "turnaround": "character turnaround sheet with front, side and back views",
          "expressions": "expression sheet with six different emotions", "outfit": "outfit design with accessories"}

def g_character(rng, d):
    """Character design with sheet formats of increasing complexity."""
    facet = rng.choice(["single", "full_body"] if d <= 2 else list(SHEETS))
    core = rng.choice(ARCHETYPES)
    if d >= 3: core += f" wearing {a(color(rng, d) + ' ' + rng.choice(['scarf', 'cloak', 'jacket', 'hat', 'backpack']))}"
    extras = [SHEETS[facet]] + (["plain background", "consistent design"] if d >= 4 else [])
    ar = "16:9" if facet in ("turnaround", "expressions") else None
    return dict(core=core, facet=facet, keywords=[], extras=extras, ar=ar)

# ---------------- creative / posters ----------------
POSTER_TYPES = {"movie": "a movie poster", "concert": "a concert poster", "travel": "a vintage travel poster",
                "public_info": "a public information poster", "festival": "a film festival poster",
                "product": "a product launch poster", "circus": "a vintage circus poster",
                "exhibition": "an art exhibition poster", "science": "a science fair poster"}
TITLE_A = ["Midnight", "Silent", "Golden", "Last", "Electric", "Northern", "Hidden", "Crimson", "Endless", "Paper",
           "Velvet", "Iron", "Little", "Wild", "Blue"]
TITLE_B = ["Harbor", "Garden", "Signal", "Summer", "Frontier", "Orchestra", "Tides", "Parade", "Echo", "Lanterns",
           "Orchard", "Station", "Comet", "River", "Mountain"]
TAGLINES = ["Coming Soon", "One Night Only", "Opening June 14", "Live in Berlin", "Free Entry", "Since 1924",
            "Book Now", "Tickets at the Door", "Visit Kyoto", "Save Water", "The Future Is Here", "Summer 2026"]

def title(rng):
    """Random invented title (never a real product)."""
    t = f"{rng.choice(TITLE_A)} {rng.choice(TITLE_B)}"
    return t.upper() if rng.random() < 0.5 else ("The " + t if rng.random() < 0.3 else t)

def g_posters(rng, d):
    """Posters with titles, taglines and imagery."""
    facet = rng.choice(list(POSTER_TYPES))
    t = title(rng)
    core = f'{POSTER_TYPES[facet]} with the title "{t}"'
    if d >= 3: core += f' and the tagline "{rng.choice(TAGLINES)}"'
    if d >= 2: core += f", featuring {rng.choice(GENERAL_SUBJECTS + CONCEPT['environment'])}"
    extras = [rng.choice(["bold typography", "elegant serif typography", "hand-lettered title"])] if d >= 4 else []
    if d >= 5: extras.append("clear visual hierarchy")
    return dict(core=core, facet=facet, keywords=[], extras=extras)

# ---------------- creative / comics ----------------
COMIC_STORIES = ["a cat trying to catch a laser dot", "a robot learning to bake a cake", "two kids building a treehouse",
                 "a penguin who wants to fly", "a wizard losing his hat in the wind", "a dog waiting for the mail",
                 "an astronaut discovering a flower on the moon", "a snail racing a turtle",
                 "a chef whose soup comes alive", "a ghost who is afraid of the dark"]
COMIC_LINES = ["Not again!", "Wait for me!", "I made it!", "Is it lunch yet?", "Trust me.", "Oops.", "Look up!",
               "Best day ever!", "Shh...", "Did you hear that?"]

def g_comics(rng, d):
    """Comics: panel count grows with difficulty, speech bubbles from d>=3."""
    n = {1: 1, 2: 2, 3: 3, 4: 4, 5: rng.choice([4, 6])}[d]
    story = rng.choice(COMIC_STORIES)
    core = (f"a single-panel cartoon of {story}" if n == 1 else
            f"{a(str(n) + '-panel comic strip')} about {story}")
    lines = rng.sample(COMIC_LINES, 2)
    if d >= 3: core += f', with a speech bubble saying "{lines[0]}"'
    if d >= 5: core += f' and a second speech bubble saying "{lines[1]}"'
    ar = rng.choice(["16:9", "4:3"]) if n in (2, 3, 4) and rng.random() < 0.6 else None
    return dict(core=core, facet=f"panels_{n}", keywords=[str(n)] if n > 1 else [], extras=[], ar=ar)

# ---------------- creative / text rendering ----------------
TEXT_WORDS = ["OPEN", "SALE", "EXIT", "HELLO", "BAKERY", "HOTEL", "PHARMACY", "LIBRARY", "WELCOME", "STOP", "JAZZ",
              "TAXI", "RAMEN", "FLOWERS", "BOOKS", "SUSHI", "CINEMA", "MARKET", "DANGER", "LOVE"]
TEXT_PHRASES = ["Fresh Bread Daily", "Welcome Home", "Quiet Please", "Coffee & Books", "Happy Birthday Emma",
                "Gate 42", "Est. 1987", "Good Vibes Only", "Mind the Step", "Back in 5 Minutes", "Live Music Tonight",
                "Handmade with Love", "The Rusty Anchor", "Lucky Noodle House", "Moonlight Motel", "Studio 9",
                "Nihonga Tea Room", "Open 24 Hours", "Wet Paint", "Hello, World", "Sunday Brunch", "Room 404",
                "Thank You!", "No Swimming", "Best Dad Ever"]
TEXT_ZH = ["欢迎光临", "新年快乐", "咖啡", "书店", "茶馆", "早安", "春天来了", "禁止吸烟", "面馆", "谢谢惠顾",
           "福", "生日快乐", "小心地滑", "营业中", "山水", "你好世界"]
TEXT_LONG = ["The best way to predict the future is to create it", "Every journey begins with a single step",
             "Please remove your shoes before entering", "Today's special: tomato soup and grilled cheese",
             "All you need is a good book and a cup of tea", "Slow down, enjoy the view",
             "Knowledge is the only treasure that grows when shared"]
TEXT_TEMPLATES = {"sign": 'a wooden sign that reads "{t}"', "street_sign": 'a street sign with the text "{t}"',
                  "storefront": 'a shop storefront with the name "{t}" painted above the door',
                  "neon": 'a neon sign glowing with the words "{t}" on a brick wall',
                  "book_cover": 'a hardcover book titled "{t}" lying on a desk',
                  "label": 'a glass jar with a label that says "{t}"',
                  "handwritten": 'a handwritten note on lined paper that says "{t}"',
                  "chalkboard": 'a café chalkboard with "{t}" written in chalk',
                  "tshirt": 'a person wearing a t-shirt printed with "{t}"',
                  "cake": 'a birthday cake with "{t}" written in icing', "sand": '"{t}" written in the sand on a beach',
                  "mug": 'a ceramic mug printed with "{t}"', "banner": 'a banner hanging between two trees reading "{t}"'}

def g_text(rng, d):
    """Text rendering: single words -> phrases -> Chinese / bilingual / multi-element -> long sentences."""
    tkey = rng.choice(list(TEXT_TEMPLATES))
    if d == 1:
        return dict(core=TEXT_TEMPLATES[tkey].format(t=rng.choice(TEXT_WORDS)), facet=tkey, keywords=[])
    if d == 2:
        return dict(core=TEXT_TEMPLATES[tkey].format(t=rng.choice(TEXT_PHRASES)), facet=tkey, keywords=[])
    if d == 3:
        if rng.random() < 0.4:
            return dict(core=TEXT_TEMPLATES[tkey].format(t=rng.choice(TEXT_ZH)), facet=f"chinese_{tkey}", keywords=[])
        return dict(core=TEXT_TEMPLATES[tkey].format(t=rng.choice(TEXT_PHRASES)), facet=tkey, keywords=[],
                    extras=[rng.choice(["legible typography", "clean lettering", "correct spelling"])])
    if d == 4:
        if rng.random() < 0.5:
            en, zh = rng.choice(TEXT_PHRASES), rng.choice(TEXT_ZH)
            return dict(core=f'a restaurant sign with "{zh}" on the top line and "{en}" below it', facet="bilingual",
                        keywords=[])
        t1, t2 = rng.sample(TEXT_PHRASES, 2)
        return dict(core=f'a storefront named "{t1}" with a small sign in the window saying "{t2}"',
                    facet="multi_text", keywords=[])
    if rng.random() < 0.5:
        return dict(core=TEXT_TEMPLATES[rng.choice(["handwritten", "chalkboard", "banner", "book_cover"])].format(
                    t=rng.choice(TEXT_LONG)), facet="long_text", keywords=[], extras=["every letter legible"])
    items = rng.sample(["Espresso $3", "Latte $4", "Matcha $5", "Croissant $3", "Bagel $2", "Tea $2", "Mocha $4"], 3)
    return dict(core=f'a café chalkboard menu listing "{items[0]}", "{items[1]}" and "{items[2]}"',
                facet="menu_multi_text", keywords=[], extras=["neat chalk lettering"])

# ---------------- creative / surreal ----------------
SURREAL = {"material_swap": "{o} made entirely of {m}", "floating": "{big} floating above {place}",
           "portal_door": "a door standing alone in {place}, opening onto {place2}",
           "scale_shift": "{o} as large as a house in {place}", "hybrid": "{an} with the wings of a butterfly",
           "living_city": "a city built on the back of a giant {an2}",
           "melting": "clocks melting over the branches of a tree in {place}",
           "impossible_stairs": "an endless staircase spiraling up into the clouds",
           "inverted_world": "an upside-down reflection of {place} in the sky",
           "weather_indoor": "a small raincloud raining inside a living room"}
SURREAL_MATS = ["clouds", "glass", "stacked books", "ice", "candy", "flowers", "origami paper", "smoke", "water",
                "knitted yarn"]
PLACES = ["a desert", "a quiet suburban street", "an empty beach", "a snowy forest", "a city square", "a wheat field",
          "the ocean", "a library"]

def g_surreal(rng, d):
    """Surreal scenes with impossible combinations."""
    facet = rng.choice(list(SURREAL))
    p1, p2 = rng.sample(PLACES, 2)
    core = SURREAL[facet].format(o=a(rng.choice(OBJECTS)), m=rng.choice(SURREAL_MATS),
                                 big=a(rng.choice(["whale", "lighthouse", "piano", "train", "island"])), place=p1,
                                 place2=p2, an=a(rng.choice(LAND_ANIMALS)), an2=rng.choice(["turtle", "whale", "elephant"]))
    extras = [rng.choice(["dreamlike atmosphere", "surrealism", "magical realism"])] if d >= 2 else []
    if d >= 4: extras.append(rng.choice(["photorealistic details", "coherent lighting", "precise perspective"]))
    return dict(core=core, facet=facet, keywords=[], extras=extras)

# ---------------- creative / storytelling ----------------
STORY_SCENES = [("a child discovering a hidden door behind a bookshelf", "mystery"),
                ("an old sailor returning home to a dog waiting on the pier", "drama"),
                ("a young woman opening a letter with long-awaited news", "drama"),
                ("a knight kneeling before a tiny dragon offering a flower", "fantasy"),
                ("the last passenger on a night bus with a mysterious package", "mystery"),
                ("a robot watering the last plant on a ruined earth", "sci_fi"),
                ("a grandmother teaching her grandson to fold dumplings", "slice_of_life"),
                ("a group of explorers finding an ancient city in the jungle", "adventure"),
                ("a dog proudly presenting a stolen sausage to its owner", "comedy"),
                ("a musician playing violin alone on a sinking ship", "drama"),
                ("two rival chefs realizing they used the same secret recipe", "comedy"),
                ("a lighthouse keeper spotting a strange light on the horizon", "mystery"),
                ("a girl releasing a paper lantern for her lost friend", "drama"),
                ("an astronaut seeing earth rise for the first time", "sci_fi")]
STORY_TONES = ["a sense of quiet anticipation", "bittersweet mood", "hopeful atmosphere", "playful tone",
               "eerie silence", "triumphant mood"]

def g_story(rng, d):
    """Scenes that imply a narrative (before/after, emotion, tension)."""
    core, facet = rng.choice(STORY_SCENES)
    extras = []
    if d >= 2: extras.append(rng.choice(STORY_TONES))
    if d >= 3: extras.append(rng.choice(["expressive body language", "environmental storytelling details",
                                         "clear focal point"]))
    if d >= 5: extras.append(rng.choice(["cinematic composition", "dramatic lighting"]))
    return dict(core=core, facet=facet, keywords=[], extras=extras)

# ---------------- creative / unusual compositions ----------------
UNUSUAL = {"miniature_world": "tiny people exploring a giant {o} as if it were a mountain",
           "worms_eye": "{s} seen from a worm's-eye view, towering overhead",
           "fisheye": "{s} seen through a fisheye lens with extreme distortion",
           "recursive": "a painter painting a picture of themselves painting the same picture",
           "cross_section": "a cross-section of {x} showing everything inside",
           "tilt_shift": "{scene} photographed with a tilt-shift effect so it looks like a miniature",
           "object_as_landscape": "a landscape where the hills are made of {mat}",
           "reflection_only": "{s} visible only as a reflection in a puddle",
           "bird_overhead": "a perfectly overhead view of {s} with long shadows",
           "through_object": "{scene} seen through a keyhole"}
CROSS_SECTIONS = ["an old house", "a submarine", "an anthill", "a cruise ship", "a volcano", "a beehive", "a pyramid"]

def g_unusual(rng, d):
    """Unusual viewpoints and compositions."""
    facet = rng.choice(list(UNUSUAL))
    core = UNUSUAL[facet].format(o=rng.choice(OBJECTS), s=rng.choice(GENERAL_SUBJECTS),
                                 x=rng.choice(CROSS_SECTIONS), scene=rng.choice(["a busy train station",
                                 "a harbor", "a city intersection", "a stadium", "a village square"]),
                                 mat=rng.choice(["folded fabric", "bread", "sleeping cats", "crumpled paper"]))
    extras = [rng.choice(["striking composition", "unexpected perspective", "playful framing"])] if d >= 3 else []
    if d >= 5: extras.append(rng.choice(["highly detailed", "coherent perspective"]))
    return dict(core=core, facet=facet, keywords=[], extras=extras)

print("creative generators ready")

creative generators ready


## 10. Generator registry, quotas and `make_record`

* `GENERATORS` maps `(dimension, subdimension)` → generator function.
* `compute_quotas()` turns dimension/sub-dimension weights into exact integer quotas summing to the pool size.
* `make_record()` samples difficulty-dependent style, prompting format, optional extras and aspect ratio, renders the
  prompt and returns the full metadata record.

In [29]:
# Registry: (dimension, subdimension) -> generator
GENERATORS = {
    ("alignment", "counting"): g_counting, ("alignment", "colors"): g_colors, ("alignment", "shapes"): g_shapes,
    ("alignment", "materials"): g_materials, ("alignment", "multiple_objects"): g_multiple_objects,
    ("alignment", "actions"): g_actions, ("alignment", "spatial_relation"): g_spatial,
    ("alignment", "composition_layout"): g_layout,
    ("visual_quality", "textures"): g_textures, ("visual_quality", "transparent_materials"): g_transparent,
    ("visual_quality", "reflective_materials"): g_reflective, ("visual_quality", "skin"): g_skin,
    ("visual_quality", "fur"): g_fur, ("visual_quality", "foliage"): g_foliage,
    ("visual_quality", "low_light"): g_low_light, ("visual_quality", "fine_detail"): g_fine_detail,
    ("aesthetics", "composition"): g_composition, ("aesthetics", "lighting"): g_lighting,
    ("aesthetics", "color_harmony"): g_color_harmony, ("aesthetics", "portraits"): g_portraits,
    ("aesthetics", "landscapes"): g_landscapes, ("aesthetics", "cinematic_framing"): g_cinematic,
    ("aesthetics", "illustration_styles"): g_illustration,
    ("real_world", "architecture"): g_architecture, ("real_world", "interiors"): g_interiors,
    ("real_world", "vehicles"): g_vehicles, ("real_world", "clothing"): g_clothing,
    ("real_world", "everyday_objects"): g_everyday, ("real_world", "cultural_scenes"): g_cultural,
    ("real_world", "physical_interactions"): g_physics,
    ("creative", "concept_art"): g_concept, ("creative", "character_design"): g_character,
    ("creative", "posters"): g_posters, ("creative", "comics"): g_comics, ("creative", "text_rendering"): g_text,
    ("creative", "surreal"): g_surreal, ("creative", "storytelling"): g_story,
    ("creative", "unusual_compositions"): g_unusual,
}

def compute_quotas(total):
    """Exact integer quota per (dimension, subdimension) using largest-remainder rounding."""
    raw = {}
    for dim, dw in DIMENSION_WEIGHTS.items():
        subs = [s for (d_, s) in GENERATORS if d_ == dim]
        sw = {s: SUBDIM_WEIGHT_OVERRIDES.get(s, 1.0) for s in subs}
        for s in subs:
            raw[(dim, s)] = total * dw * sw[s] / sum(sw.values())
    quotas = {k: int(v) for k, v in raw.items()}
    for k in sorted(raw, key=lambda k: raw[k] - quotas[k], reverse=True)[: total - sum(quotas.values())]:
        quotas[k] += 1
    return quotas

NO_CAMERA_SUBDIMS = {"cinematic_framing", "portraits", "landscapes", "skin", "low_light", "fine_detail"}  # own shots

def make_record(rng, dim, sub, d):
    """Generate one fully-annotated template prompt."""
    out = GENERATORS[(dim, sub)](rng, d)
    style = out.get("style") or weighted(rng, SUBDIM_STYLE_POOL.get(sub, DIM_STYLE_POOL[dim]))
    fmt = weighted(rng, FORMAT_BY_DIFFICULTY[d])
    extras = list(out.get("extras", []))                                  # mandatory extras
    covered = " ".join(extras).lower()
    pools = [p for p in DIM_OPTIONAL_POOLS[dim]
             if not (p == "camera" and (style not in PHOTO_STYLES or sub in NO_CAMERA_SUBDIMS))
             and not (p == "mood" and re.search(r"atmosphere|mood|tone", covered))
             and not (p == "lighting" and re.search(r"light|sun|lit", covered))]
    n_opt = rng.randint(*EXTRAS_BY_DIFFICULTY[d]) if fmt != "short" else 0
    for cat in rng.sample(pools, min(n_opt, len(pools))):                 # at most one extra per category
        extras.append(rng.choice(OPTIONAL_POOLS[cat]))
    prompt = render(rng, out["core"], extras, style, fmt)
    return {
        "prompt": prompt,
        "dimension": dim,
        "subdimension": sub,
        "facet": out["facet"],
        "difficulty": d,
        "style": style,
        "aspect_ratio": out.get("ar") or weighted(rng, SUBDIM_AR.get(sub, DEFAULT_AR)),
        "language": "en",
        "prompt_format": fmt,
        "hard": d >= 4 or (sub in HARD_SUBDIMS and d >= 3),
        "keywords": out.get("keywords", []),
        "source": "template",
        "prompt_template": prompt,
        "generator_version": GENERATOR_VERSION,
    }

QUOTAS = compute_quotas(MASTER_POOL_SIZE)
print(f"{len(GENERATORS)} sub-dimensions, quotas sum = {sum(QUOTAS.values())}")
for dim in DIMENSION_WEIGHTS:
    print(f"  {dim:15s}", {s: q for (d_, s), q in QUOTAS.items() if d_ == dim})

38 sub-dimensions, quotas sum = 100000
  alignment       {'counting': 3930, 'colors': 3023, 'shapes': 3023, 'materials': 3023, 'multiple_objects': 3023, 'actions': 3023, 'spatial_relation': 3930, 'composition_layout': 3023}
  visual_quality  {'textures': 2250, 'transparent_materials': 2250, 'reflective_materials': 2250, 'skin': 2250, 'fur': 2250, 'foliage': 2250, 'low_light': 2250, 'fine_detail': 2250}
  aesthetics      {'composition': 2572, 'lighting': 2572, 'color_harmony': 2572, 'portraits': 2572, 'landscapes': 2572, 'cinematic_framing': 2572, 'illustration_styles': 2572}
  real_world      {'architecture': 2572, 'interiors': 2572, 'vehicles': 2571, 'clothing': 2571, 'everyday_objects': 2571, 'cultural_scenes': 2571, 'physical_interactions': 2571}
  creative        {'concept_art': 2222, 'character_design': 2222, 'posters': 2222, 'comics': 2222, 'text_rendering': 4445, 'surreal': 2222, 'storytelling': 2222, 'unusual_compositions': 2222}


## 11. Smoke test — samples from every sub-dimension

Prints one sample per sub-dimension at difficulties 1, 3 and 5 so we can eyeball grammar and diversity
before generating the full pool.

In [30]:
# Print one prompt per sub-dimension at three difficulty levels (visual QA of the templates)
rng_demo = random.Random(123)                              # separate RNG: does not affect the master pool
for (dim, sub) in GENERATORS:
    print(f"--- {dim} / {sub}")
    for d in (1, 3, 5):
        r = make_record(rng_demo, dim, sub, d)
        print(f"  d={d} [{r['facet']}|{r['style']}|{r['prompt_format']}|{r['aspect_ratio']}] {r['prompt']}")

--- alignment / counting
  d=1 [count_1|photographic|imperative|9:16] Render a single squirrel on a rocky hillside. Style: candid photo, natural light.
  d=3 [count_6|photographic|descriptive|1:1] A photorealistic photograph of six silver coffee mugs on a park bench. Sharp focus. Photorealistic, professional photography.
  d=5 [multi_count_2types|digital_illustration|imperative|3:4] Create an image of exactly three parrots and four dogs in a grassy meadow, high dynamic range. Style: digital painting, painterly.
--- alignment / colors
  d=1 [single_color|3d_render|tags|1:1] a purple mushroom, stylized 3D, clay-like materials
  d=3 [binding_2|3d_render|natural|4:3] A 3D render of a purple lemon and a coral violin on a concrete floor, crisp details.
  d=5 [counterfactual_color|3d_render|imperative|16:9] Render an indigo banana next to a pink paper crane on a red tablecloth, natural colors. Style: 3D render, soft global illumination.
--- alignment / shapes
  d=1 [single_shape|photographic|

## 12. Generate the 100k master pool

For each sub-dimension we draw difficulties from `DIFFICULTY_WEIGHTS` and generate until the quota of **unique**
prompts (normalized exact de-duplication across the whole pool) is reached. Any shortfall is reported.

**Cache:** if `master_pool_100k.jsonl` already exists, §12–§14 do **not** recompute it: the saved pool is loaded and
an *already calculated* message is printed (set `FORCE_RECOMPUTE = True` to regenerate). Loading a pool produced by a
different `GENERATOR_VERSION` raises an error instead of silently mixing versions.

In [31]:
# Load the master pool if already calculated, otherwise generate it with global exact de-duplication
POOL_CACHED = MASTER_POOL_PATH.exists() and not FORCE_RECOMPUTE
if POOL_CACHED:
    pool = read_jsonl(MASTER_POOL_PATH)                    # saved pool, already decontaminated/split/ranked
    versions = {r["generator_version"] for r in pool}
    if versions != {GENERATOR_VERSION}:
        raise RuntimeError(f"{MASTER_POOL_PATH} was built with {versions}, notebook is {GENERATOR_VERSION}: "
                           f"set FORCE_RECOMPUTE = True to regenerate it.")
    print(f"Master pool already calculated -> loaded {len(pool):,} prompts from {MASTER_POOL_PATH} "
          f"(set FORCE_RECOMPUTE = True to regenerate)")
else:
    # Generate the master pool with global exact de-duplication
    rng = random.Random(MASTER_SEED)                           # master RNG -> deterministic pool
    seen, pool, shortfall = set(), [], {}
    t0 = time.time()
    for (dim, sub), quota in QUOTAS.items():
        made, attempts = 0, 0
        while made < quota and attempts < quota * 30:          # generous retry budget per sub-dimension
            attempts += 1
            d = weighted(rng, DIFFICULTY_WEIGHTS)
            rec = make_record(rng, dim, sub, d)
            key = norm_key(rec["prompt"])
            if key in seen:                                    # drop exact duplicates
                continue
            seen.add(key)
            pool.append(rec)
            made += 1
        if made < quota:
            shortfall[f"{dim}/{sub}"] = quota - made
    print(f"Generated {len(pool):,} unique prompts in {time.time() - t0:.1f}s")
    print("Shortfall per sub-dimension:", shortfall or "none")

Task was destroyed but it is pending!
task: <Task pending name='Task-489' coro=<Event.wait() running at C:\Users\jjmca\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\asyncio\locks.py:213> wait_for=<Future pending cb=[Task.task_wakeup()]>>
Task was destroyed but it is pending!
task: <Task pending name='Task-508' coro=<Event.wait() running at C:\Users\jjmca\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\asyncio\locks.py:213> wait_for=<Future pending cb=[Task.task_wakeup()]>>
Task was destroyed but it is pending!
task: <Task pending name='Task-524' coro=<Event.wait() running at C:\Users\jjmca\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\asyncio\locks.py:213> wait_for=<Future pending cb=[Task.task_wakeup()]>>
Task was destroyed but it is pending!
task: <Task pending name='Task-538' coro=<Event.wait() running at C:\Users\jjmca\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\asyncio\locks.py:213> wait_for=<Future pending cb=[Ta

Master pool already calculated -> loaded 100,000 prompts from data\phase1\master_pool_100k.jsonl (set FORCE_RECOMPUTE = True to regenerate)


## 13. Decontamination against Qwen-Image-Bench

Qwen-Image-Bench is **evaluation only**. This section:

1. Downloads the benchmark file from Hugging Face at a **pinned revision** (`BENCH_REVISION`). The prompts only exist
   inside `qwen_image_bench_hf_v0518.jsonl` (185 MB; it also holds per-model image paths and judge outputs), which
   stays in the local HF cache.
2. Extracts the 1,000 bilingual prompts (`ID, prompt_en, prompt_cn, dims_en, dims_cn`) to
   `data/qwen_image_bench/prompts.jsonl` and records provenance (repo, revision, SHA-256) in `source.json`.
   Phase 2 reuses this file for the baseline benchmark.
3. Removes from the pool any prompt that is an exact normalized match of a benchmark prompt (EN or ZH), or whose
   n-grams overlap the benchmark by ≥ `DECON_THRESHOLD` (word 6-grams for English, character 8-grams for Chinese).
   The same check is applied again to the LLM rewrites/translations in §19.

In [32]:
# Download + extract Qwen-Image-Bench prompts, then remove any overlap from the synthetic pool
from huggingface_hub import hf_hub_download

# 1) Download the pinned benchmark file (kept in the local HF cache, not copied into the repo)
bench_raw_path = hf_hub_download(BENCH_REPO, BENCH_FILE, repo_type="dataset", revision=BENCH_REVISION)
bench_raw = read_jsonl(bench_raw_path)

# 2) Extract only the prompt fields and serialize them with provenance
bench = [{"ID": int(r["ID"]), "prompt_en": r["prompt_en"], "prompt_cn": r["prompt_cn"],
          "dims_en": r["dims_en"], "dims_cn": r["dims_cn"]} for r in bench_raw]
write_jsonl(BENCH_PROMPTS_PATH, bench)
bench_source = {"repo": BENCH_REPO, "revision": BENCH_REVISION, "file": BENCH_FILE, "n_prompts": len(bench),
                "sha256": hashlib.sha256(Path(bench_raw_path).read_bytes()).hexdigest(),
                "downloaded_utc": datetime.now(timezone.utc).isoformat(), "usage": "EVALUATION ONLY"}
(BENCH_DIR / "source.json").write_text(json.dumps(bench_source, indent=2), encoding="utf-8")
print(f"Extracted {len(bench)} bench prompts (EN + ZH) -> {BENCH_PROMPTS_PATH}")

# 3) Build the overlap index (exact keys + n-grams, both languages)
CJK_RE = re.compile(r"[一-鿿]")

def is_chinese(text):
    """Mostly-Chinese text (English prompts may contain short quoted Chinese strings)."""
    chars = re.sub(r"\s", "", text)
    return bool(chars) and len(CJK_RE.findall(chars)) / len(chars) >= 0.3

def ngrams(text):
    """Word n-grams for English, character n-grams for Chinese (no word boundaries)."""
    t = norm_key(text)
    if is_chinese(t):
        c, n = re.sub(r"\s", "", t), DECON_CHAR_NGRAM
        return {c[i:i + n] for i in range(max(0, len(c) - n + 1))}
    w, n = t.split(), DECON_WORD_NGRAM
    return {" ".join(w[i:i + n]) for i in range(max(0, len(w) - n + 1))}

bench_keys, bench_grams = {}, {}
for b in bench:
    for text in (b["prompt_en"], b["prompt_cn"]):
        bench_keys[norm_key(text)] = b["ID"]
        for g in ngrams(text):
            bench_grams.setdefault(g, b["ID"])

def bench_overlap(text):
    """(kind, closest_bench_id, overlap_fraction); kind is 'exact', 'ngram' or None (= clean)."""
    k = norm_key(text)
    if k in bench_keys:
        return "exact", bench_keys[k], 1.0
    g = ngrams(text)
    hits = [bench_grams[x] for x in g if x in bench_grams]
    frac = round(len(hits) / len(g), 3) if g else 0.0
    closest = Counter(hits).most_common(1)[0][0] if hits else None
    return ("ngram" if frac >= DECON_THRESHOLD else None), closest, frac

# 4) Filter the pool (only when it was freshly generated; a cached pool is already decontaminated)
if POOL_CACHED:
    print("Master pool already calculated -> decontamination already applied, see decontamination_report.json")
else:
    # 4) Filter the pool and serialize the report (every removed prompt is kept in the report)
    decon = {"bench": bench_source, "threshold": DECON_THRESHOLD, "word_ngram": DECON_WORD_NGRAM,
             "char_ngram": DECON_CHAR_NGRAM, "pool_before": len(pool), "removed": [], "overlap_hist": Counter()}
    kept = []
    for r in pool:
        kind, bid, frac = bench_overlap(r["prompt"])
        decon["overlap_hist"][f"{min(frac, 0.999):.1f}"] += 1           # distribution of overlap fractions
        if kind:
            decon["removed"].append({**r, "decon_kind": kind, "bench_id": bid, "overlap": frac})
        else:
            kept.append(r)
    pool = kept
    decon["pool_after"] = len(pool)
    decon["overlap_hist"] = dict(sorted(decon["overlap_hist"].items()))
    (DATA_DIR / "decontamination_report.json").write_text(json.dumps(decon, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Removed {len(decon['removed'])} prompts | pool: {decon['pool_before']:,} -> {decon['pool_after']:,}")
    print("Overlap-fraction histogram:", decon["overlap_hist"])
    for x in decon["removed"][:10]:
        print(f"  removed ({x['decon_kind']}, bench #{x['bench_id']}, {x['overlap']}): {x['prompt']}")

Extracted 1000 bench prompts (EN + ZH) -> data\qwen_image_bench\prompts.jsonl
Master pool already calculated -> decontamination already applied, see decontamination_report.json


## 14. Ids, seeds, stratified splits and nested pilot ranking

* `id` = hash of the template prompt, `seed` = deterministic per-prompt seed (reused by later phases for fixed noise).
* Inside every stratum `(dimension, subdimension, difficulty)` prompts are shuffled; the in-stratum index `i`
  gives the split (`SPLIT_PATTERN[i % 20]` → 90/5/5) and a global `pool_rank` ordered by `i / stratum_size`.
* Consequence: **any prefix of `pool_rank` (20k pilot, 50k, 100k) is stratified and has exactly the same split
  assignment**, so the dataset can grow in Phase 15 without leaking prompts between splits.
* Skipped when the pool was loaded from disk (it already carries ids, seeds, splits and ranks).

In [33]:
# Assign ids/seeds, stratified splits and a nested stratified ranking; serialize the master pool
if POOL_CACHED:
    print(f"Master pool already calculated -> ids/splits/pool_rank loaded from {MASTER_POOL_PATH}")
else:
    rng_split = random.Random(MASTER_SEED + 1)                 # dedicated RNG for splitting/ranking
    for r in pool:
        h = sha(r["prompt_template"], 16)
        r["id"] = f"p1-{h[:12]}"                               # stable id derived from the template text
        r["seed"] = int(h[:8], 16)                             # deterministic generation seed for later phases

    strata = defaultdict(list)                                 # group by (dimension, subdimension, difficulty)
    for r in pool:
        strata[(r["dimension"], r["subdimension"], r["difficulty"])].append(r)

    ranking = []
    for key in sorted(strata):
        group = strata[key]
        rng_split.shuffle(group)
        offset = rng_split.random()                            # random phase so strata do not all start with "train"
        for i, r in enumerate(group):
            r["split"] = SPLIT_PATTERN[(i + int(offset * len(SPLIT_PATTERN))) % len(SPLIT_PATTERN)]
            ranking.append(((i + rng_split.random()) / len(group), r["id"], r))
    ranking.sort(key=lambda x: (x[0], x[1]))
    for rank, (_, _, r) in enumerate(ranking):
        r["pool_rank"] = rank
    pool = [r for _, _, r in ranking]                          # pool ordered by pool_rank

    assert len({r["id"] for r in pool}) == len(pool), "id collision"
    n = write_jsonl(MASTER_POOL_PATH, pool)
    print(f"Saved {n:,} prompts to {MASTER_POOL_PATH}")
    print("Split distribution (full pool):", Counter(r["split"] for r in pool))

Master pool already calculated -> ids/splits/pool_rank loaded from data\phase1\master_pool_100k.jsonl


## 15. Select and serialize the 20k pilot (template version)

The pilot is the first `PILOT_SIZE` prompts by `pool_rank`. **Cache:** if `pilot_20k/templates.jsonl` exists (and the
pool was not regenerated) it is loaded instead of recomputed, and checked against the pool. We also write the split registry, which documents the four
splits of the plan (TRAIN / VALIDATION / INTERNAL TEST / QWEN-IMAGE-BENCH).

In [34]:
# Pilot = first PILOT_SIZE prompts by pool_rank (stratified, nested inside the 100k pool); loaded if already saved
PILOT_CACHED = POOL_CACHED and PILOT_TEMPLATES_PATH.exists()   # a regenerated pool always regenerates the pilot
if PILOT_CACHED:
    pilot = read_jsonl(PILOT_TEMPLATES_PATH)
    if [r["id"] for r in pilot] != [r["id"] for r in pool[:PILOT_SIZE]]:
        raise RuntimeError(f"{PILOT_TEMPLATES_PATH} does not match the first {PILOT_SIZE} prompts of the pool: "
                           f"set FORCE_RECOMPUTE = True to regenerate both.")
    print(f"Pilot already calculated -> loaded {len(pilot):,} prompts from {PILOT_TEMPLATES_PATH}")
else:
    pilot = [dict(r) for r in pool[:PILOT_SIZE]]
    write_jsonl(PILOT_TEMPLATES_PATH, pilot)
    print(f"Pilot: {len(pilot):,} prompts -> {PILOT_TEMPLATES_PATH}")
print("Split:", Counter(r["split"] for r in pilot))
print("Dimension:", Counter(r["dimension"] for r in pilot))
print("Difficulty:", sorted(Counter(r["difficulty"] for r in pilot).items()))

# Registry of the four splits required by the plan
SPLITS_REGISTRY = {
    "phase": PHASE,
    "splits": {
        "train": {"source": "synthetic (this notebook)", "usage": "bridge pretraining, healing, speed distillation"},
        "validation": {"source": "synthetic (this notebook)", "usage": "model selection, early stopping"},
        "internal_test": {"source": "synthetic (this notebook)", "usage": "final internal evaluation only"},
        "qwen_image_bench": {"source": f"hf://datasets/{BENCH_REPO}@{BENCH_REVISION}/{BENCH_FILE}",
                             "local_prompts": str(BENCH_PROMPTS_PATH), "n_prompts": len(bench),
                             "usage": "EVALUATION ONLY — never used for training",
                             "pool_decontaminated": True, "llm_outputs_decontaminated": True},
    },
    "split_rule": "stratified by (dimension, subdimension, difficulty); SPLIT_PATTERN 18:1:1; nested by pool_rank",
}
(DATA_DIR / "splits_registry.json").write_text(json.dumps(SPLITS_REGISTRY, indent=2, ensure_ascii=False), encoding="utf-8")

Pilot already calculated -> loaded 20,000 prompts from data\phase1\pilot_20k\templates.jsonl
Split: Counter({'train': 17991, 'internal_test': 1007, 'validation': 1002})
Dimension: Counter({'alignment': 5196, 'creative': 4000, 'aesthetics': 3603, 'visual_quality': 3602, 'real_world': 3599})
Difficulty: [(1, 2653), (2, 4926), (3, 6181), (4, 4162), (5, 2078)]


945

## 16. LLM rewriting — task selection, instructions and validation

**Why we need it.** The teacher can supervise any prompt, so the pipeline would *run* on templates alone — but the
healing would not transfer well to real prompts:

* **Distillation happens in text-embedding space.** The student is matched to the teacher *for a given prompt
  embedding*. ~40 template sentence shapes cover a narrow region of that space; a pruned model healed only on them can
  match the teacher on template phrasing and still regress on natural phrasing.
* **Qwen-Image-Bench does not look like our templates.** It is 50 % Chinese (and the leaderboard is computed on the
  Chinese prompts), half of it is long prompts (EN median ≈ 39 words), written naturally with several constraints each.
  Our template pilot is 100 % English, ≈ 125 characters on average. Qwen-Image is bilingual: without Chinese prompts in
  healing, pruning can silently damage Chinese-prompt behaviour and we would only discover it at Phase 8.
* **The plan requires it:** Phase 1 lists *LLM rewriting* among the generation methods.

The known risk — the LLM dropping or changing constraints ("three" → "a few") — is handled by the automatic validation
below, with fallback to the template prompt.

**How it works.**

* A deterministic hash of each prompt id decides its LLM task:
  `translate_zh` (10 %), `rewrite_en` (35 % of non-`tags` prompts, tag prompts keep their format), or none.
* The LLM must preserve every constraint; we **validate automatically**:
  quoted text must appear verbatim, alignment keywords (counts, colors, relations) must survive in English rewrites,
  length must stay within bounds, translations must actually be Chinese.
* Failed outputs fall back to the template prompt; nothing is discarded (raw outputs are logged).

In [35]:
# LLM task assignment, instruction prompts and validators
SYSTEM_REWRITE = (
    "You rewrite prompts for a text-to-image model. Rules: keep every visual constraint exactly "
    "(object counts, colors, materials, shapes, spatial relations, actions, layout, visual style). "
    "Copy any text inside double quotes character-for-character and keep it inside double quotes. "
    "Do not add new objects, text or constraints. Vary vocabulary and sentence structure naturally. "
    "Output only the rewritten prompt on a single line, without any preamble or explanation.")
SYSTEM_TRANSLATE = (
    "You translate prompts for a text-to-image model into natural Simplified Chinese, the way a native speaker "
    "would write an image-generation prompt. Keep every visual constraint exactly (counts, colors, materials, "
    "shapes, spatial relations, actions, layout, visual style). Text inside double quotes is text to be rendered in "
    "the image: do NOT translate it, copy it character-for-character inside double quotes. "
    "Output only the translation on a single line, without any preamble or explanation.")
LENGTH_HINTS = ["Keep roughly the same length.", "Make it slightly more descriptive.", "Make it more concise."]

def llm_task(r):
    """Deterministic task assignment from the prompt id."""
    u = int(sha(r["id"] + "llm", 8), 16) / 0xFFFFFFFF
    if u < LLM_ZH_TRANSLATE_FRACTION:
        return "translate_zh"
    if r["prompt_format"] != "tags" and u < LLM_ZH_TRANSLATE_FRACTION + LLM_EN_REWRITE_FRACTION:
        return "rewrite_en"
    return None

def build_request(r):
    """Chat request (system + user) for one record."""
    task = llm_task(r)
    if task == "rewrite_en":
        hint = LENGTH_HINTS[int(sha(r["id"] + "len", 4), 16) % len(LENGTH_HINTS)]
        return {"id": r["id"], "task": task, "system": SYSTEM_REWRITE, "user": f"{hint}\nPrompt: {r['prompt_template']}"}
    return {"id": r["id"], "task": task, "system": SYSTEM_TRANSLATE, "user": f"Prompt: {r['prompt_template']}"}

QUOTED = re.compile(r'"([^"]+)"')
CJK = re.compile(r"[一-鿿]")

def clean_output(text):
    """Strip think-blocks, labels and wrapping quotes; keep the first non-empty line."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    line = next((l.strip() for l in text.splitlines() if l.strip()), "")
    line = re.sub(r"^(rewritten prompt|prompt|translation|译文|提示词)\s*[:：]\s*", "", line, flags=re.I)
    if len(line) > 1 and line[0] == line[-1] and line[0] in "\"'“”" and line.count('"') == 2:
        line = line[1:-1]
    return line.strip()

def validate(r, task, out):
    """Return (ok, reason) for an LLM output."""
    src = r["prompt_template"]
    if not out:
        return False, "empty"
    for q in QUOTED.findall(src):
        if q not in out:
            return False, f"quoted_text_missing:{q}"
    if task == "rewrite_en":
        if CJK.search(out) and not CJK.search(src):
            return False, "unexpected_cjk"
        if not 0.5 <= len(out) / len(src) <= 3.0:
            return False, "length_ratio"
        low = out.lower()
        for kw in r["keywords"]:
            if kw.lower() not in low:
                return False, f"keyword_missing:{kw}"
        if re.match(r"^(sure|here is|here's|certainly)", low):
            return False, "preamble"
    else:
        stripped = QUOTED.sub("", out)
        if len(CJK.findall(stripped)) < 0.3 * len(re.sub(r"\s", "", stripped)):
            return False, "not_chinese"
        if len(out) > 2.5 * len(src):
            return False, "too_long"
    return True, "ok"

REQUESTS = [build_request(r) for r in pilot if llm_task(r)]
print(f"LLM requests: {len(REQUESTS):,}", Counter(q["task"] for q in REQUESTS))
print("Example:", json.dumps(REQUESTS[0], ensure_ascii=False)[:400])

LLM requests: 8,154 Counter({'rewrite_en': 6146, 'translate_zh': 2008})
Example: {"id": "p1-56b1997ef89b", "task": "rewrite_en", "system": "You rewrite prompts for a text-to-image model. Rules: keep every visual constraint exactly (object counts, colors, materials, shapes, spatial relations, actions, layout, visual style). Copy any text inside double quotes character-for-character and keep it inside double quotes. Do not add new objects, text or constraints. Vary vocabulary an


## 17. Run it on vLLM on top of Modal for speed

**Why vLLM on Modal.**

* **Volume:** ~8k generations for the 20k pilot (and ~4× more for the remaining 80k of the 100k expansion). vLLM's continuous batching
  processes them in minutes on a single GPU instead of one-by-one generation.
* **Open model, data stays with us:** `Qwen/Qwen3-8B` is an open instruct model (strong in both English and Chinese);
  prompts are not sent to a third-party API. A hosted API would also work and be cheap at this volume — the rest of the
  notebook does not depend on how the outputs are produced.

**How it works.**

Defines a Modal class that loads `LLM_MODEL` with vLLM on a `LLM_GPU` and processes batches of chat requests.
Weights are cached in a persistent Modal volume. (`serialized=True` lets Modal ship the class from a notebook.)

In [36]:
# Modal + vLLM rewriter definition (nothing runs remotely until the next cell)
import modal

app = modal.App("nihonga-phase1-rewriter")                                   # Modal app name
vllm_image = modal.Image.debian_slim(python_version="3.13") \
    .pip_install("vllm==0.30.0") \
    .env({"VLLM_USE_FLASHINFER_SAMPLER": "0"})                               # FlashInfer sampler JIT-compiles with nvcc (absent in debian_slim)
hf_cache = modal.Volume.from_name("nihonga-hf-cache", create_if_missing=True)  # persistent HF weights cache
_MODEL, _TEMP = LLM_MODEL, LLM_TEMPERATURE                                    # captured by value for the container

@app.cls(image=vllm_image, gpu=LLM_GPU, volumes={"/root/.cache/huggingface": hf_cache}, timeout=3600,
         max_containers=LLM_MAX_CONTAINERS, scaledown_window=120, serialized=True)
class Rewriter:
    @modal.enter()
    def load(self):
        """Load the model once per container."""
        from vllm import LLM, SamplingParams
        self.llm = LLM(model=_MODEL, max_model_len=4096, gpu_memory_utilization=0.90)
        self.params = SamplingParams(temperature=_TEMP, top_p=0.9, max_tokens=320, seed=0)

    @modal.method()
    def run(self, batch):
        """Generate one output per request in the batch."""
        convs = [[{"role": "system", "content": q["system"]}, {"role": "user", "content": q["user"]}] for q in batch]
        outs = self.llm.chat(convs, self.params, chat_template_kwargs={"enable_thinking": False}, use_tqdm=False)
        return [{"id": q["id"], "task": q["task"], "output": o.outputs[0].text} for q, o in zip(batch, outs)]

print(f"Modal app ready: model={LLM_MODEL}, gpu={LLM_GPU}, batch={LLM_BATCH_SIZE}")

Modal app ready: model=Qwen/Qwen3-8B, gpu=L40S, batch=256


## 18. Run the LLM rewriting on Modal (resumable)
```
~10% (≈2,000)	translated to Simplified Chinese (any format, including tags)
~30.5% (≈6,100)	English paraphrase: 35% of the 17,448 non-tags prompts
~59.5% (≈11,900)	template text only
```
Results are appended to `llm_raw.jsonl` as soon as each batch returns, so an interrupted run loses nothing:
re-running this cell only sends the requests whose ids are not yet in the raw file.

In [37]:
# Execute the rewriting remotely, appending every raw result immediately (resumable)
RAW_PATH = PILOT_DIR / "llm_raw.jsonl"
by_id = {r["id"]: r for r in pilot}
done = {row["id"] for row in read_jsonl(RAW_PATH)} if RAW_PATH.exists() else set()
todo = [q for q in REQUESTS if q["id"] not in done]
print(f"{len(done):,} already done, {len(todo):,} to run")

if RUN_LLM_REWRITE and todo:
    batches = [todo[i:i + LLM_BATCH_SIZE] for i in range(0, len(todo), LLM_BATCH_SIZE)]
    t0 = time.time()
    # Jupyter already runs an event loop, so Modal must be driven with its async API (async with / .map.aio)
    with modal.enable_output():
        async with app.run():
            with open(RAW_PATH, "a", encoding="utf-8") as f:
                k = 0
                async for results in Rewriter().run.map.aio(batches, order_outputs=False):
                    for res in results:
                        out = clean_output(res["output"])
                        ok, reason = validate(by_id[res["id"]], res["task"], out)
                        f.write(json.dumps({**res, "clean": out, "ok": ok, "reason": reason, "model": LLM_MODEL,
                                            "ts": datetime.now(timezone.utc).isoformat()}, ensure_ascii=False) + "\n")
                    f.flush()
                    k += 1
                    print(f"batch {k}/{len(batches)} done ({time.time() - t0:.0f}s)")
elif not RUN_LLM_REWRITE:
    print("RUN_LLM_REWRITE=False -> skipping; the pilot will contain template prompts only.")

0 already done, 8,154 to run
✓ Initialized. View run at 
https://modal.com/apps/jjmcarrascosa/main/ap-vKqkmc9kvecOAD1flAs98e
- Initializing...
Building image im-gA83RDTLuclwesLM9V4sPt
\ Creating objects....
=> Step 0: FROM base
m/ Creating objects...
=> Step 1: ENV VLLM_USE_FLASHINFER_SAMPLER=0
Saving image...cts....
Image saved, took 1.92s
m| Creating objects...
Built image im-gA83RDTLuclwesLM9V4sPt in 2.89s


\ Creating objects....
/ Creating objects...Rewriter.*...
- Creating objects...*.
└── 🔨 Created function Rewriter.*.
✓ Created objects.
└── 🔨 Created function Rewriter.*.
\ Running app.....
┌─────────────────────────────────────────────────────────────────────────────┐
│ Rewriter.run ----------------------------------------  0/32 -:--:--         │
/ Running app...──────────────────────────────┘
┌─────────────────────────────────────────────────────────────────────────────┐
│ Rewriter.run ----------------------------------------  0/32 -:--:--         │
\ Running app...───────────

## 19. Merge LLM outputs and write the final pilot splits

Accepted LLM outputs are re-checked against Qwen-Image-Bench (a rewrite/translation could drift towards a benchmark
prompt); clean ones replace `prompt` (the original stays in `prompt_template`), `language`/`source` are updated,
and the final `train / validation / internal_test` files are written.

In [38]:
# Merge validated LLM outputs into the pilot and serialize the final splits
raw = read_jsonl(RAW_PATH) if RAW_PATH.exists() else []
accepted, bench_rejected = {}, []
for row in raw:                                              # last accepted output per id wins
    if not row["ok"]:
        continue
    kind, bid, frac = bench_overlap(row["clean"])            # decontaminate LLM outputs too
    if kind:
        bench_rejected.append({"id": row["id"], "prompt": row["clean"], "kind": kind, "bench_id": bid, "overlap": frac})
        continue
    accepted[row["id"]] = row
(PILOT_DIR / "llm_bench_rejections.json").write_text(json.dumps(bench_rejected, indent=2, ensure_ascii=False),
                                                     encoding="utf-8")
print(f"LLM outputs rejected for bench overlap: {len(bench_rejected)}")
final = []
for r in pilot:
    r = dict(r)
    if r["id"] in accepted:
        row = accepted[r["id"]]
        r["prompt"] = row["clean"]
        r["source"] = "llm_translate" if row["task"] == "translate_zh" else "llm_rewrite"
        r["language"] = "zh" if row["task"] == "translate_zh" else "en"
        r["llm_model"] = row["model"]
    final.append(r)

# Guard: LLM outputs must not create duplicates
dupes = [k for k, c in Counter(norm_key(r["prompt"]) for r in final).items() if c > 1]
print(f"Duplicate prompts after merge: {len(dupes)}")

for split in ("train", "validation", "internal_test"):
    rows = [r for r in final if r["split"] == split]
    write_jsonl(PILOT_DIR / f"{split}.jsonl", rows)
    print(f"{split:14s} {len(rows):6,} -> {PILOT_DIR / (split + '.jsonl')}")

if raw:
    print("LLM validation:", Counter(r["ok"] for r in raw),
          Counter(r["reason"].split(":")[0] for r in raw if not r["ok"]).most_common(8))
print("Source:", Counter(r["source"] for r in final), "| Language:", Counter(r["language"] for r in final))

LLM outputs rejected for bench overlap: 0
Duplicate prompts after merge: 0
train          17,991 -> data\phase1\pilot_20k\train.jsonl
validation      1,002 -> data\phase1\pilot_20k\validation.jsonl
internal_test   1,007 -> data\phase1\pilot_20k\internal_test.jsonl
LLM validation: Counter({True: 7817, False: 337}) [('keyword_missing', 290), ('length_ratio', 32), ('unexpected_cjk', 9), ('quoted_text_missing', 6)]
Source: Counter({'template': 12183, 'llm_rewrite': 5813, 'llm_translate': 2004}) | Language: Counter({'en': 17996, 'zh': 2004})


## 20. Statistics, sanity checks and manifest

Distribution tables per split, dimension, sub-dimension, difficulty, style, format, aspect ratio and language;
random examples of the final dataset; manifest with SHA-256 of every artifact, environment and git commit.

In [39]:
# Compute and save statistics of the final pilot
def dist(rows, key):
    """Normalized distribution of a metadata field."""
    c = Counter(r[key] for r in rows)
    return {str(k): round(v / len(rows), 4) for k, v in c.most_common()}

stats = {"phase": PHASE, "pilot_size": len(final), "master_pool_size": len(pool),
         "by_split": dict(Counter(r["split"] for r in final))}
for key in ("dimension", "subdimension", "difficulty", "style", "prompt_format", "aspect_ratio", "language",
            "source", "hard"):
    stats[key] = dist(final, key)
stats["facets_per_subdimension"] = {s: len({r["facet"] for r in final if r["subdimension"] == s})
                                    for s in sorted({r["subdimension"] for r in final})}
stats["mean_prompt_chars"] = round(sum(len(r["prompt"]) for r in final) / len(final), 1)
(DATA_DIR / "stats.json").write_text(json.dumps(stats, indent=2, ensure_ascii=False), encoding="utf-8")

for key in ("dimension", "difficulty", "prompt_format", "language", "source", "hard"):
    print(f"{key:14s}", stats[key])
print("mean prompt length (chars):", stats["mean_prompt_chars"])

# Sanity checks on split integrity
ids = {s: {r["id"] for r in final if r["split"] == s} for s in ("train", "validation", "internal_test")}
assert not (ids["train"] & ids["validation"]) and not (ids["train"] & ids["internal_test"]), "split leakage"
assert all(r["split"] == p["split"] for r, p in zip(final, pool[:PILOT_SIZE])), "pilot/pool split mismatch"
print("\nSplit integrity checks passed.")

# Random examples from the final dataset
for r in random.Random(7).sample(final, 12):
    print(f"[{r['split']}|{r['dimension']}/{r['subdimension']}/{r['facet']}|d{r['difficulty']}|{r['language']}|"
          f"{r['source']}] {r['prompt']}")

dimension      {'alignment': 0.2598, 'creative': 0.2, 'aesthetics': 0.1802, 'visual_quality': 0.1801, 'real_world': 0.1799}
difficulty     {'3': 0.309, '2': 0.2463, '4': 0.2081, '1': 0.1326, '5': 0.1039}
prompt_format  {'natural': 0.3475, 'descriptive': 0.2608, 'short': 0.1434, 'tags': 0.1276, 'imperative': 0.1207}
language       {'en': 0.8998, 'zh': 0.1002}
source         {'template': 0.6091, 'llm_rewrite': 0.2907, 'llm_translate': 0.1002}
hard           {'False': 0.6091, 'True': 0.3908}
mean prompt length (chars): 120.8

Split integrity checks passed.
[train|aesthetics/landscapes/mountains|d3|en|template] A photorealistic photograph of a jagged snow-capped mountain range in autumn colors with a winding path in the foreground, high dynamic range.
[train|visual_quality/transparent_materials/liquid|d1|en|template] a glass of sparkling water, DSLR photo, high resolution
[train|aesthetics/illustration_styles/concept_art|d3|en|llm_rewrite] A conceptual illustration of a red bicycle set aga

In [40]:
# Write the manifest: file hashes, row counts, environment, git commit (full traceability)
def file_sha256(p):
    """SHA-256 of a file."""
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL).strip()
except Exception:
    commit = None

files = {}
for p in sorted(list(DATA_DIR.rglob("*")) + list(BENCH_DIR.rglob("*"))):
    if p.is_file() and p.name != "manifest.json":
        rows = sum(1 for _ in open(p, encoding="utf-8")) if p.suffix == ".jsonl" else None
        files[p.relative_to(DATA_DIR.parent).as_posix()] = {"sha256": file_sha256(p), "bytes": p.stat().st_size, "rows": rows}

manifest = {"phase": PHASE, "generator_version": GENERATOR_VERSION, "created_utc": datetime.now(timezone.utc).isoformat(),
            "git_commit": commit, "python": sys.version, "platform": platform.platform(),
            "llm_model": LLM_MODEL if RUN_LLM_REWRITE else None, "files": files}
(DATA_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
for k, v in files.items():
    print(f"{k:44s} rows={v['rows']!s:>7} bytes={v['bytes']:>11,}  sha256={v['sha256'][:12]}")

phase1/config.json                           rows=   None bytes=      1,880  sha256=3dc7192fbb0a
phase1/decontamination_report.json           rows=   None bytes=        576  sha256=b1db8ede6248
phase1/master_pool_100k.jsonl                rows= 100000 bytes= 65,193,477  sha256=d358101292e8
phase1/pilot_20k/internal_test.jsonl         rows=   1007 bytes=    683,730  sha256=75e18e3232e3
phase1/pilot_20k/llm_bench_rejections.json   rows=   None bytes=          2  sha256=4f53cda18c2b
phase1/pilot_20k/llm_raw.jsonl               rows=   8154 bytes=  3,697,470  sha256=7313c72e4fdc
phase1/pilot_20k/templates.jsonl             rows=  20000 bytes= 13,034,786  sha256=2962796d8294
phase1/pilot_20k/train.jsonl                 rows=  17991 bytes= 12,055,102  sha256=56ed184aaed2
phase1/pilot_20k/validation.jsonl            rows=   1002 bytes=    677,683  sha256=eb6b0366b9bf
phase1/splits_registry.json                  rows=   None bytes=        974  sha256=e3dd7c7eb46f
phase1/stats.json             